# GeoSemTwin-X — prototype, comparative studies and digital-twin dashboard (v2)

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Log in to Hugging Face when asked.

| Part | What it produces | Time on T4 |
|---|---|---|
| A · TUM RGB-D | SLAM accuracy vs ORB-SLAM2 / ElasticFusion / DVO-SLAM, SLAM component ablation, office twin | ~25 min |
| B · Staer warehouse | SLAM configuration sweep, semantic fusion vs 3 conventional methods + 3 ablations, pose-error robustness, detector-threshold sensitivity, warehouse twin | ~35 min |
| C · Synthetic warehouse (optional) | exact-ground-truth benchmark incl. change detection across two sessions and exploration | ~40 min |
| Dashboard | `GeoSemTwinX_Dashboard.html`: 3D twin, asset registry, spatial queries, change log, versions, all comparative studies, performance | — |

**What changed since v1:** Staer evaluation counts only assets a detector can realistically see (≤ 8 m, ≥ 0.2 % of the image,
in ≥ 2 keyframes); Staer geometry uses the depth sensor's own poses; forklifts / pallet trucks must touch the floor;
the map is an uncertainty-weighted fused point map (Open3D 0.20's TSDF returned empty maps on Colab); detections are
computed once and reused by every study.

In [ ]:
!nvidia-smi -L
!pip -q install open3d plotly sam2 huggingface_hub tabulate
import sys, torch
print("Python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA", torch.cuda.is_available())

## 1 · Install the GeoSemTwin-X package

In [ ]:
!mkdir -p /content/geosemtwinx /content/checkpoints /content/results/comparative /content/results/figures

In [ ]:
%%writefile /content/geosemtwinx/__init__.py
"""GeoSemTwin-X prototype: geometry-grounded semantic SLAM -> persistent digital twin."""
__version__ = "0.1.0"

In [ ]:
%%writefile /content/geosemtwinx/synthetic.py
"""Synthetic warehouse RGB-D generator (CPU, Open3D ray casting).

Produces a sequence with ground-truth camera poses, depth, RGB (procedural
texture so photometric odometry has gradients), per-pixel instance and semantic
labels, plus an optional "session B" with known asset changes for evaluating
change detection.  Used to test every non-neural module of GeoSemTwin-X with
exact ground truth.
"""
import cv2
import numpy as np
import open3d as o3d

CLASSES = ["floor", "wall", "rack", "pallet", "cardboard box", "forklift", "barrel"]
CLASS_ID = {c: i for i, c in enumerate(CLASSES)}
ASSET_CLASSES = ["pallet", "cardboard box", "forklift", "barrel"]

BASE_COLOR = {
    "floor": (0.55, 0.55, 0.52), "wall": (0.80, 0.78, 0.72), "rack": (0.15, 0.30, 0.65),
    "pallet": (0.62, 0.45, 0.25), "cardboard box": (0.72, 0.55, 0.33),
    "forklift": (0.90, 0.70, 0.05), "barrel": (0.20, 0.45, 0.25),
}


def _box(size, center, yaw=0.0):
    m = o3d.geometry.TriangleMesh.create_box(*size)
    m.translate(-np.asarray(size) / 2.0)
    R = o3d.geometry.get_rotation_matrix_from_xyz((0, 0, yaw))
    m.rotate(R, center=(0, 0, 0))
    m.translate(center)
    return m


def _cyl(radius, height, center):
    m = o3d.geometry.TriangleMesh.create_cylinder(radius, height, resolution=24)
    m.translate(center)
    return m


def _merge(parts):
    out = o3d.geometry.TriangleMesh()
    for p in parts:
        out += p
    return out


def make_forklift(center, yaw):
    cx, cy, _ = center
    R = o3d.geometry.get_rotation_matrix_from_xyz((0, 0, yaw))
    parts = [
        _box((1.1, 1.9, 1.0), (0, 0, 0.55)),          # body
        _box((1.0, 0.6, 1.1), (0, -0.55, 1.6)),       # cabin roof block
        _box((0.9, 0.12, 2.2), (0, 1.0, 1.1)),        # mast
        _box((0.12, 1.0, 0.06), (-0.3, 1.5, 0.08)),   # fork L
        _box((0.12, 1.0, 0.06), (0.3, 1.5, 0.08)),    # fork R
    ]
    m = _merge(parts)
    m.rotate(R, center=(0, 0, 0))
    m.translate((cx, cy, 0))
    return m


def make_rack(x0, y0, length, depth=1.0, levels=3, height=4.0, along_x=True):
    parts = []
    n_up = int(length // 2.0) + 1
    for k in range(n_up):
        for d in (0, depth):
            if along_x:
                c = (x0 + k * (length / (n_up - 1)), y0 + d, height / 2)
            else:
                c = (x0 + d, y0 + k * (length / (n_up - 1)), height / 2)
            parts.append(_box((0.08, 0.08, height), c))
    for lv in range(levels):
        z = 0.3 + lv * (height - 0.4) / (levels - 1) if levels > 1 else 0.3
        if along_x:
            parts.append(_box((length, depth, 0.05), (x0 + length / 2, y0 + depth / 2, z)))
        else:
            parts.append(_box((depth, length, 0.05), (x0 + depth / 2, y0 + length / 2, z)))
    return _merge(parts)


def default_layout(seed=0):
    """List of objects: dict(name, cls, mesh_fn args). Assets have stable names."""
    rng = np.random.default_rng(seed)
    objs = []
    W, D, H = 24.0, 16.0, 6.0
    objs.append(dict(name="floor", cls="floor", mesh=_box((W, D, 0.02), (W / 2, D / 2, -0.01))))
    objs.append(dict(name="ceiling", cls="wall", mesh=_box((W, D, 0.1), (W / 2, D / 2, H + 0.05))))
    objs.append(dict(name="wall_s", cls="wall", mesh=_box((W, 0.1, H), (W / 2, -0.05, H / 2))))
    objs.append(dict(name="wall_n", cls="wall", mesh=_box((W, 0.1, H), (W / 2, D + 0.05, H / 2))))
    objs.append(dict(name="wall_w", cls="wall", mesh=_box((0.1, D, H), (-0.05, D / 2, H / 2))))
    objs.append(dict(name="wall_e", cls="wall", mesh=_box((0.1, D, H), (W + 0.05, D / 2, H / 2))))
    # racks along north and south walls and a middle row
    objs.append(dict(name="rack_1", cls="rack", mesh=make_rack(2, 14.4, 20)))
    objs.append(dict(name="rack_2", cls="rack", mesh=make_rack(2, 0.6, 20)))
    objs.append(dict(name="rack_3", cls="rack", mesh=make_rack(4, 7.5, 12)))
    # pallets with boxes, forklifts, barrels in aisles
    pallet_spots = [(3.0, 4.0), (6.0, 3.2), (9.5, 4.2), (13.0, 3.5), (17.0, 4.0), (20.5, 3.4),
                    (3.5, 12.0), (8.0, 12.4), (12.5, 11.8), (16.5, 12.3), (21.0, 11.9)]
    for i, (x, y) in enumerate(pallet_spots):
        yaw = rng.uniform(-0.3, 0.3)
        objs.append(dict(name=f"pallet_{i:02d}", cls="pallet", mesh=_box((1.2, 1.0, 0.15), (x, y, 0.075), yaw)))
        if i % 2 == 0:
            objs.append(dict(name=f"box_{i:02d}", cls="cardboard box",
                             mesh=_box((0.6, 0.5, 0.5), (x, y, 0.15 + 0.25), yaw)))
    for i, (x, y, yaw) in enumerate([(11.0, 5.6, 0.4), (19.0, 10.0, -1.2), (5.5, 9.8, 2.0)]):
        objs.append(dict(name=f"forklift_{i:02d}", cls="forklift", mesh=make_forklift((x, y, 0), yaw)))
    for i, (x, y) in enumerate([(22.5, 6.0), (22.4, 9.5), (1.5, 6.5), (14.5, 9.6)]):
        objs.append(dict(name=f"barrel_{i:02d}", cls="barrel", mesh=_cyl(0.3, 0.9, (x, y, 0.0))))
    return objs


def apply_changes(objs, seed=1):
    """Session B: move, remove and add assets. Returns new list + GT change list."""
    out, changes = [], []
    for o in objs:
        if o["name"] == "box_04":                       # removed
            changes.append(dict(name="box_04", type="removed"))
            continue
        if o["name"] == "pallet_03":                    # moved 1.6 m
            m = o3d.geometry.TriangleMesh(o["mesh"]); m.translate((1.6, 0.0, 0))
            out.append(dict(o, mesh=m)); changes.append(dict(name="pallet_03", type="moved"))
            continue
        if o["name"] == "forklift_01":                  # moved and rotated
            m = o3d.geometry.TriangleMesh(o["mesh"])
            c = m.get_center()
            m.rotate(o3d.geometry.get_rotation_matrix_from_xyz((0, 0, 0.9)), center=c)
            m.translate((-2.0, 0.6, 0))
            out.append(dict(o, mesh=m)); changes.append(dict(name="forklift_01", type="moved"))
            continue
        if o["name"] == "barrel_02":                    # removed
            changes.append(dict(name="barrel_02", type="removed"))
            continue
        out.append(o)
    out.append(dict(name="barrel_new", cls="barrel", mesh=_cyl(0.3, 0.9, (10.0, 9.6, 0.0))))
    changes.append(dict(name="barrel_new", type="added"))
    out.append(dict(name="box_new", cls="cardboard box", mesh=_box((0.6, 0.5, 0.5), (15.0, 5.0, 0.25), 0.2)))
    changes.append(dict(name="box_new", type="added"))
    return out, changes


def look_at(eye, target, up=(0, 0, 1)):
    """Camera-to-world pose, optical frame (x right, y down, z forward)."""
    eye, target, up = map(np.asarray, (eye, target, up))
    z = target - eye; z = z / np.linalg.norm(z)
    x = np.cross(z, up); x = x / np.linalg.norm(x)
    y = np.cross(z, x)
    T = np.eye(4); T[:3, 0], T[:3, 1], T[:3, 2], T[:3, 3] = x, y, z, eye
    return T


def default_trajectory(n=360, height=1.4, loop_frames=720):
    """Loop around the middle rack row (one loop = loop_frames frames, ~0.055 m/frame at 30 fps),
    looking ahead with gentle head turns. Returns the first n poses."""
    t = np.arange(n) * 2 * np.pi / loop_frames
    cx, cy, ax, ay = 12.0, 8.0, 8.6, 3.3
    poses = []
    for k, tt in enumerate(t):
        eye = np.array([cx + ax * np.cos(tt), cy + ay * np.sin(tt), height + 0.05 * np.sin(5 * tt)])
        ahead = np.array([cx + ax * np.cos(tt + 0.25), cy + ay * np.sin(tt + 0.25), 0.9])
        d = ahead - eye
        side = np.cross([0, 0, 1], d / np.linalg.norm(d))
        target = ahead + 2.5 * np.sin(3 * tt) * side   # look left/right
        poses.append(look_at(eye, target))
    return poses


class SyntheticWarehouse:
    def __init__(self, objs, width=640, height=480, fx=525.0, fy=525.0, cx=319.5, cy=239.5,
                 depth_noise=True, max_depth=20.0, seed=0):
        self.objs, self.W, self.H = objs, width, height
        self.K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]], float)
        self.depth_noise, self.max_depth = depth_noise, max_depth
        self.rng = np.random.default_rng(seed)
        self.scene = o3d.t.geometry.RaycastingScene()
        self.gid_to_obj = {}
        self.instance_names = ["background"]
        for o in objs:
            gid = self.scene.add_triangles(o3d.t.geometry.TriangleMesh.from_legacy(o["mesh"]))
            self.gid_to_obj[gid] = o
            if o["cls"] in ASSET_CLASSES:
                self.instance_names.append(o["name"])
        self.inst_id = {n: i for i, n in enumerate(self.instance_names)}
        u, v = np.meshgrid(np.arange(width), np.arange(height))
        self.dirs_cam = np.stack([(u - cx) / fx, (v - cy) / fy, np.ones_like(u, float)], -1).reshape(-1, 3)

    def render(self, T_wc):
        R, t = T_wc[:3, :3], T_wc[:3, 3]
        dirs = self.dirs_cam @ R.T
        rays = np.concatenate([np.repeat(t[None], len(dirs), 0), dirs], 1).astype(np.float32)
        ans = self.scene.cast_rays(o3d.core.Tensor(rays))
        z = ans["t_hit"].numpy().reshape(self.H, self.W)       # planar depth (dir has z=1)
        gid = ans["geometry_ids"].numpy().reshape(self.H, self.W)
        normals = ans["primitive_normals"].numpy().reshape(self.H, self.W, 3)
        valid = np.isfinite(z) & (z < self.max_depth)
        pts = (t + dirs.reshape(self.H, self.W, 3) * np.where(valid, z, 0)[..., None])
        sem = np.zeros((self.H, self.W), np.int32)
        inst = np.zeros((self.H, self.W), np.int32)
        base = np.zeros((self.H, self.W, 3))
        for g, o in self.gid_to_obj.items():
            m = (gid == g) & valid
            if not m.any():
                continue
            sem[m] = CLASS_ID[o["cls"]]
            if o["cls"] in ASSET_CLASSES:
                inst[m] = self.inst_id[o["name"]]
            base[m] = BASE_COLOR[o["cls"]]
        # procedural texture: multi-frequency pattern on world coords + per-object tint
        p = pts
        tex = (0.55 + 0.18 * np.sin(3.1 * p[..., 0] + 1.7 * p[..., 2]) * np.sin(2.3 * p[..., 1] - 0.9 * p[..., 2])
               + 0.12 * np.sin(11.0 * p[..., 0] + 7.0 * p[..., 1] + 5.0 * p[..., 2])
               + 0.10 * ((np.floor(p[..., 0] * 2) + np.floor(p[..., 1] * 2) + np.floor(p[..., 2] * 2)) % 2))
        # high-frequency surface detail (hashed value noise at 4 cm and 12 cm) -> trackable corners
        for cell, amp in ((25.0, 0.22), (8.0, 0.18)):
            q = np.floor(p * cell)
            hsh = np.sin(q[..., 0] * 12.9898 + q[..., 1] * 78.233 + q[..., 2] * 37.719) * 43758.5453
            tex = tex + amp * (hsh - np.floor(hsh) - 0.5)
        light = 0.55 + 0.45 * np.clip(np.abs(normals @ np.array([0.3, 0.2, 0.93])), 0, 1)
        rgb = np.clip(base * tex[..., None] * light[..., None] * 1.25, 0, 1)
        rgb[~valid] = 0
        rgb = cv2.GaussianBlur(rgb.astype(np.float32), (0, 0), 0.8)          # lens blur / anti-aliasing
        rgb = (rgb * 255 + self.rng.normal(0, 2.0, rgb.shape)).clip(0, 255).astype(np.uint8)
        depth = np.where(valid, z, 0.0).astype(np.float32)
        if self.depth_noise:
            sigma = 0.0012 + 0.0019 * (np.maximum(depth, 0.4) - 0.4) ** 2
            depth = depth + self.rng.normal(0, 1, depth.shape) * sigma
            depth[self.rng.random(depth.shape) < 0.01] = 0.0
            depth[~valid] = 0.0
            depth = np.round(depth * 1000) / 1000.0                      # mm quantisation
        return dict(rgb=rgb, depth=depth.astype(np.float32), semantic=sem, instance=inst)


def build_sequence(session="A", n_frames=360, seed=0, scale=1.0):
    objs = default_layout(seed)
    changes = []
    if session == "B":
        objs, changes = apply_changes(objs)
    poses = default_trajectory(n_frames)
    sim = SyntheticWarehouse(objs, width=int(640 * scale), height=int(480 * scale), fx=525.0 * scale,
                             fy=525.0 * scale, cx=320 * scale - 0.5, cy=240 * scale - 0.5,
                             seed=seed + (7 if session == "B" else 0))
    inst_class = {sim.inst_id[o["name"]]: o["cls"] for o in objs if o["cls"] in ASSET_CLASSES}
    return sim, poses, inst_class, changes

In [ ]:
%%writefile /content/geosemtwinx/data.py
"""Dataset adapters. Every sequence exposes the same interface:

    seq.K            3x3 intrinsics
    len(seq)         number of frames
    seq.frame(i)     dict(rgb uint8 HxWx3, depth float32 metres, [instance int32], [semantic int32])
    seq.gt_poses     list of 4x4 camera-to-world (optical frame) or None
    seq.timestamps   seconds
    seq.instance_class  {instance id: class name} for GT evaluation (optional)
"""
import json
import os
import re
import subprocess
import tarfile
import urllib.request
from pathlib import Path

import cv2
import numpy as np


def quat_to_R(qw, qx, qy, qz):
    q = np.array([qw, qx, qy, qz], float)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


# --------------------------------------------------------------------------- synthetic
class SyntheticSequence:
    def __init__(self, session="A", n_frames=360, stride=1, seed=0, scale=1.0):
        from .synthetic import build_sequence
        self.sim, poses, self.instance_class, self.changes = build_sequence(session, n_frames, seed, scale)
        self.idx = list(range(0, n_frames, stride))
        self.gt_poses = [poses[i] for i in self.idx]
        self.K = self.sim.K.copy()
        self.timestamps = [i / 30.0 for i in self.idx]
        self.instance_names = self.sim.instance_names
        self._cache = {}

    def __len__(self):
        return len(self.idx)

    def frame(self, i):
        if i not in self._cache:
            self._cache[i] = self.sim.render(self.gt_poses[i])
        return self._cache[i]


# --------------------------------------------------------------------------- TUM RGB-D
TUM_INTRINSICS = {
    "freiburg1": (517.3, 516.5, 318.6, 255.3),
    "freiburg2": (520.9, 521.0, 325.1, 249.7),
    "freiburg3": (535.4, 539.2, 320.1, 247.6),
}


def _read_tum_list(path):
    out = []
    for line in open(path):
        if line.startswith("#") or not line.strip():
            continue
        p = line.split()
        out.append((float(p[0]), p[1:]))
    return out


def _associate(a, b, max_dt=0.02):
    tb = np.array([t for t, _ in b])
    pairs, used = [], set()
    for ta, va in a:
        j = int(np.argmin(np.abs(tb - ta)))
        if abs(tb[j] - ta) < max_dt and j not in used:
            used.add(j)
            pairs.append((ta, va, b[j][1]))
    return pairs


class TUMSequence:
    """name e.g. 'rgbd_dataset_freiburg1_desk'. Downloads (~0.4 GB) on first use."""
    URL = "https://cvg.cit.tum.de/rgbd/dataset/{fr}/{name}.tgz"

    def __init__(self, name="rgbd_dataset_freiburg1_desk", root="/content/data/tum", stride=1, max_frames=None):
        fr = re.search(r"freiburg\d", name).group(0)
        root = Path(root); root.mkdir(parents=True, exist_ok=True)
        seq_dir = root / name
        if not seq_dir.exists():
            tgz = root / f"{name}.tgz"
            if not tgz.exists():
                print("Downloading", name, "...")
                urllib.request.urlretrieve(self.URL.format(fr=fr, name=name), tgz)
            with tarfile.open(tgz) as tf:
                tf.extractall(root)
        fx, fy, cx, cy = TUM_INTRINSICS[fr]
        self.K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]], float)
        rgb = _read_tum_list(seq_dir / "rgb.txt")
        depth = _read_tum_list(seq_dir / "depth.txt")
        pairs = _associate(rgb, depth)
        gt = _read_tum_list(seq_dir / "groundtruth.txt")
        gt_t = np.array([t for t, _ in gt])
        frames, poses = [], []
        for t, vr, vd in pairs:
            j = int(np.argmin(np.abs(gt_t - t)))
            if abs(gt_t[j] - t) > 0.02:
                continue
            tx, ty, tz, qx, qy, qz, qw = map(float, gt[j][1])
            T = np.eye(4); T[:3, :3] = quat_to_R(qw, qx, qy, qz); T[:3, 3] = (tx, ty, tz)
            frames.append((t, seq_dir / vr[0], seq_dir / vd[0])); poses.append(T)
        frames, poses = frames[::stride], poses[::stride]
        if max_frames:
            frames, poses = frames[:max_frames], poses[:max_frames]
        self.files, self.gt_poses = frames, poses
        self.timestamps = [f[0] for f in frames]
        self.instance_class = None

    def __len__(self):
        return len(self.files)

    def frame(self, i):
        _, fr, fd = self.files[i]
        rgb = cv2.cvtColor(cv2.imread(str(fr)), cv2.COLOR_BGR2RGB)
        depth = cv2.imread(str(fd), cv2.IMREAD_UNCHANGED).astype(np.float32) / 5000.0
        return dict(rgb=rgb, depth=depth)


# --------------------------------------------------------------------------- Staer warehouses
STAER_CLASS_RULES = [            # keyword in prim path -> class (first match wins)
    ("pallettruck", "pallet truck"), ("forklift", "forklift"),
    ("cardbox", "cardboard box"), ("carton", "cardboard box"), ("box", "cardboard box"),
    ("palette", "pallet"), ("pallet", "pallet"),
    ("rackshelf", "rack"), ("rack", "rack"), ("barrel", "barrel"), ("drum", "barrel"),
]


def staer_parent(prim):
    if not isinstance(prim, str) or prim.upper() == "INVALID":
        return None
    m = re.search(r"(/World/[^/]+/Prim_\d+)", prim)
    return m.group(1) if m else None


def staer_class(prim):
    p = prim.lower().replace("_", "")
    for key, cls in STAER_CLASS_RULES:
        if key in p:
            return cls
    return None


class StaerSequence:
    """Staer Robotics warehouses (gated on Hugging Face: log in first).

    Camera convention verified for this dataset: pose camera frame is x-right,
    y-up, z-backward, so optical->camera is diag(1,-1,-1). Depth is planar.
    """
    FILES = ["cam0/video.mp4", "cam0/poses.csv", "depth0/video.mkv", "depth0/depth_ranges.json",
             "instance0/video.mkv", "instance0/id_to_labels.json"]

    def __init__(self, scene="scene03", walk="001", root="/content/data/staer", scale=0.5,
                 stride=2, max_frames=None, start=0):
        from huggingface_hub import hf_hub_download
        base = f"{scene}/walkthroughs/{walk}"
        paths = {}
        for f in self.FILES:
            paths[f] = hf_hub_download("staerrobotics/warehouses", f"{base}/{f}", repo_type="dataset",
                                       local_dir=root, token=True)
        try:                                     # the depth stream has its own pose file (sensor offset)
            paths["depth0/poses.csv"] = hf_hub_download("staerrobotics/warehouses", f"{base}/depth0/poses.csv",
                                                        repo_type="dataset", local_dir=root, token=True)
        except Exception:
            paths["depth0/poses.csv"] = None
        W, H = int(1920 * scale), int(1080 * scale)
        self.K = np.array([[672.2 * scale, 0, 960 * scale], [0, 672.2 * scale, 540 * scale], [0, 0, 1]])
        ranges = json.load(open(paths["depth0/depth_ranges.json"]))
        flip = np.diag([1., -1., -1.])

        def read_poses(path):
            out = []
            for l in open(path):
                if not l.strip() or l.startswith("#"):
                    continue
                p = [float(v) for v in l.split(",")]
                T = np.eye(4); T[:3, :3] = quat_to_R(p[4], p[5], p[6], p[7]) @ flip; T[:3, 3] = p[1:4]
                out.append(T)
            return out
        cam_poses = read_poses(paths["cam0/poses.csv"])
        poses_all = cam_poses
        self.depth_pose_offset_m = 0.0
        if paths["depth0/poses.csv"]:
            dep = read_poses(paths["depth0/poses.csv"])
            if len(dep) == len(cam_poses):
                self.depth_pose_offset_m = float(np.median([np.linalg.norm(a[:3, 3] - b[:3, 3])
                                                            for a, b in zip(dep, cam_poses)]))
                poses_all = dep            # depth pixels live in the depth sensor frame
        print(f"Staer: cam0 <-> depth0 pose offset = {100 * self.depth_pose_offset_m:.1f} cm "
              f"({'using depth0 poses for geometry' if poses_all is not cam_poses else 'using cam0 poses'})")
        n = len(poses_all)
        self.idx = list(range(start, n, stride))[: max_frames or None]
        sel = set(self.idx)
        rgb = self._decode(paths["cam0/video.mp4"], W, H, "rgb24", 3, sel, "bilinear")
        inv = self._decode(paths["depth0/video.mkv"], W, H, "gray16le", 1, sel, "neighbor")
        inst = self._decode(paths["instance0/video.mkv"], W, H, "gray16le", 1, sel, "neighbor")
        self._rgb, self._inst = rgb, inst
        self._depth = []
        for k, i in enumerate(self.idx):
            v = inv[k].astype(np.float32); r = ranges[i]
            iv = r["inv_min"] + (v - 1) / 65534.0 * (r["inv_max"] - r["inv_min"])
            d = np.where(v > 0, 1.0 / np.maximum(iv, 1e-6), 0).astype(np.float32)
            self._depth.append(d)
        self.gt_poses = [poses_all[i] for i in self.idx]
        self.timestamps = [i / 20.0 for i in self.idx]
        # instance id -> parent asset + class
        labels = json.load(open(paths["instance0/id_to_labels.json"]))
        parents, self.parent_class = {}, {}
        for k, prim in labels.items():
            par = staer_parent(prim)
            if par is None:
                continue
            cls = staer_class(prim)
            parents[int(k)] = par
            if cls and par not in self.parent_class:
                self.parent_class[par] = cls
        names = sorted(set(parents.values()))
        pid = {n: i + 1 for i, n in enumerate(names)}
        lut = np.zeros(65536, np.int32)
        for k, par in parents.items():
            if k < 65536 and par in self.parent_class:      # keep only classified assets
                lut[k] = pid[par]
        self._lut = lut
        self.instance_names = ["background"] + names
        self.instance_class = {pid[n]: self.parent_class[n] for n in names if n in self.parent_class}

    @staticmethod
    def _decode(path, W, H, pix, ch, sel, flags):
        cmd = ["ffmpeg", "-v", "error", "-i", path, "-vf", f"scale={W}:{H}:flags={flags}",
               "-f", "rawvideo", "-pix_fmt", pix, "pipe:1"]
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE)
        dtype = np.uint16 if pix == "gray16le" else np.uint8
        nbytes = W * H * ch * np.dtype(dtype).itemsize
        out, k = [], 0
        while True:
            buf = proc.stdout.read(nbytes)
            if len(buf) < nbytes:
                break
            if k in sel:
                a = np.frombuffer(buf, dtype).reshape(H, W, ch) if ch > 1 else np.frombuffer(buf, dtype).reshape(H, W)
                out.append(a.copy())
            k += 1
        proc.wait()
        return out

    def __len__(self):
        return len(self.idx)

    def frame(self, i):
        return dict(rgb=self._rgb[i], depth=self._depth[i], instance=self._lut[self._inst[i]])

In [ ]:
%%writefile /content/geosemtwinx/slam.py
"""Tracking + pose-graph SLAM front/back end with per-pose uncertainty.

Prototype backend: Open3D hybrid photometric/geometric RGB-D odometry,
frame-to-keyframe tracking, radius-search loop closures and pose-graph
optimisation.  The interface (poses + covariances per frame) is the same one a
VIGS-SLAM run feeds in through `load_trajectory()`, so the downstream
GASF / mapping / twin modules are backend-agnostic.
"""
import time

import cv2
import numpy as np
import open3d as o3d

reg = o3d.pipelines.registration


def make_rgbd(frame, depth_max):
    color = o3d.geometry.Image(np.ascontiguousarray(frame["rgb"]))
    depth = o3d.geometry.Image(np.ascontiguousarray(frame["depth"].astype(np.float32)))
    return o3d.geometry.RGBDImage.create_from_color_and_depth(
        color, depth, depth_scale=1.0, depth_trunc=depth_max, convert_rgb_to_intensity=True)


def o3d_intrinsic(K, w, h):
    return o3d.camera.PinholeCameraIntrinsic(w, h, K[0, 0], K[1, 1], K[0, 2], K[1, 2])


def _rot_angle(T):
    return np.degrees(np.arccos(np.clip((np.trace(T[:3, :3]) - 1) / 2, -1, 1)))


def pose_sigma(info):
    """Scalar 1-sigma pose uncertainty (metres-equivalent) from a 6x6 information matrix."""
    try:
        cov = np.linalg.inv(info + 1e-9 * np.eye(6))
        return float(np.sqrt(max(np.trace(cov[3:, 3:]), 0)) + 0.5 * np.sqrt(max(np.trace(cov[:3, :3]), 0)))
    except np.linalg.LinAlgError:
        return 1.0


class PnPInit:
    """ORB + PnP-RANSAC relative pose (robust initialiser for dense odometry)."""

    def __init__(self, K, n_features=1500, depth_max=8.0):
        self.K = K.astype(np.float64); self.depth_max = depth_max
        self.orb = cv2.ORB_create(n_features)
        self.bf = cv2.BFMatcher(cv2.NORM_HAMMING)

    def features(self, frame):
        g = cv2.cvtColor(frame["rgb"], cv2.COLOR_RGB2GRAY)
        kp, des = self.orb.detectAndCompute(g, None)
        return kp, des, frame["depth"]

    def relative(self, fa, fb):
        """Return T_b_a (maps points of frame a into frame b) and inlier count, or (None, 0)."""
        kpa, da, depth_a = fa
        kpb, db, _ = fb
        if da is None or db is None or len(kpa) < 20 or len(kpb) < 20:
            return None, 0
        good = []
        for m in self.bf.knnMatch(da, db, k=2):
            if len(m) == 2 and m[0].distance < 0.8 * m[1].distance:
                good.append(m[0])
        if len(good) < 15:
            return None, 0
        pa = np.array([kpa[m.queryIdx].pt for m in good]); pb = np.array([kpb[m.trainIdx].pt for m in good])
        z = depth_a[pa[:, 1].astype(int), pa[:, 0].astype(int)]
        ok = (z > 0.2) & (z < self.depth_max)
        if ok.sum() < 12:
            return None, 0
        X = np.stack([(pa[ok, 0] - self.K[0, 2]) / self.K[0, 0] * z[ok],
                      (pa[ok, 1] - self.K[1, 2]) / self.K[1, 1] * z[ok], z[ok]], 1)
        res = cv2.solvePnPRansac(X, pb[ok], self.K, None, iterationsCount=200, reprojectionError=2.0,
                                 confidence=0.999, flags=cv2.SOLVEPNP_ITERATIVE)
        if not res[0] or res[3] is None or len(res[3]) < 12:
            return None, 0
        R, _ = cv2.Rodrigues(res[1])
        T = np.eye(4); T[:3, :3] = R; T[:3, 3] = res[2].ravel()
        return T, len(res[3])


def _delta(A, B):
    E = np.linalg.inv(A) @ B
    return np.linalg.norm(E[:3, 3]), _rot_angle(E)


def depth_consistency(depth_a, depth_b, K, T_b_a, stride=6, tol=0.04, depth_max=8.0):
    """Fraction of frame-a depth points that land on frame-b's depth surface after T_b_a."""
    h, w = depth_a.shape
    v, u = np.mgrid[0:h:stride, 0:w:stride]
    z = depth_a[::stride, ::stride]
    m = (z > 0.2) & (z < depth_max)
    if m.sum() < 50:
        return 0.0
    P = np.stack([(u[m] - K[0, 2]) / K[0, 0] * z[m], (v[m] - K[1, 2]) / K[1, 1] * z[m], z[m]], 1)
    Q = P @ T_b_a[:3, :3].T + T_b_a[:3, 3]
    ok = Q[:, 2] > 0.2
    uu = np.round(K[0, 0] * Q[ok, 0] / Q[ok, 2] + K[0, 2]).astype(int)
    vv = np.round(K[1, 1] * Q[ok, 1] / Q[ok, 2] + K[1, 2]).astype(int)
    inb = (uu >= 0) & (uu < w) & (vv >= 0) & (vv < h)
    if inb.sum() < 50:
        return 0.0
    zb = depth_b[vv[inb], uu[inb]]
    good = (zb > 0) & (np.abs(zb - Q[ok][inb, 2]) < tol * np.maximum(1.0, Q[ok][inb, 2] / 3))
    return float(good.sum() / m.sum())


class RGBDSLAM:
    """tracker: 'hybrid' (dense photometric+geometric), 'icp' (point-to-plane ICP on the depth cloud)
    or 'hybrid+icp' (dense result refined by ICP). use_pnp / use_loops switch components off for ablations."""

    def __init__(self, K, width, height, depth_max=6.0, kf_trans=0.20, kf_rot=12.0,
                 loop_radius=2.0, loop_min_gap=15, depth_diff_max=0.07, tracker="hybrid", use_pnp=True,
                 use_loops=True, icp_voxel=None):
        self.tracker, self.use_pnp, self.use_loops = tracker, use_pnp, use_loops
        self.icp_voxel = icp_voxel or max(0.02, depth_max / 200.0)
        self.intr = o3d_intrinsic(K, width, height)
        self.K = np.asarray(K, float)
        self.depth_max = depth_max
        self.opt = o3d.pipelines.odometry.OdometryOption(
            iteration_number_per_pyramid_level=o3d.utility.IntVector([20, 10, 5]),
            depth_diff_max=depth_diff_max, depth_min=0.2, depth_max=depth_max)
        self.jac = o3d.pipelines.odometry.RGBDOdometryJacobianFromHybridTerm()
        self.kf_trans, self.kf_rot = kf_trans, kf_rot
        self.loop_radius, self.loop_min_gap = loop_radius, loop_min_gap
        self.check_max = max(8.0, depth_max)        # range used by PnP / consistency checks
        self.pnp = PnPInit(K, depth_max=self.check_max)
        self.keyframes = []          # dict(idx, rgbd, T, feat)
        self.rel = []                # per frame: (kf index, T_kf_frame)
        self.poses, self.sigmas = [], []
        self.step_sigmas = []        # per-frame tracking uncertainty (used for gating / fusion weights)
        self.graph = reg.PoseGraph()
        self.n_loops, self.n_track_fail = 0, 0
        self.timing = []

    def _cloud(self, rgbd):
        pc = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, self.intr).voxel_down_sample(self.icp_voxel)
        pc.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=4 * self.icp_voxel, max_nn=30))
        return pc

    def _icp(self, src, tgt, init):
        a, b = self._cloud(src), self._cloud(tgt)
        T = init
        for d in (8 * self.icp_voxel, 3 * self.icp_voxel, 1.5 * self.icp_voxel):
            r = reg.registration_icp(a, b, d, T, reg.TransformationEstimationPointToPlane(),
                                     reg.ICPConvergenceCriteria(max_iteration=30))
            T = r.transformation
        info = reg.get_information_matrix_from_point_clouds(a, b, 1.5 * self.icp_voxel, T)
        return r.fitness > 0.3, T, info

    def _odom(self, src, tgt, init):
        if self.tracker == "icp":
            return self._icp(src, tgt, init)
        ok, T, info = o3d.pipelines.odometry.compute_rgbd_odometry(src, tgt, self.intr, init, self.jac, self.opt)
        if self.tracker == "hybrid+icp":
            ok2, T2, info2 = self._icp(src, tgt, T if ok else init)
            if ok2:
                return True, T2, info2
        return ok, T, info

    def track(self, frame):
        t0 = time.perf_counter()
        rgbd = make_rgbd(frame, self.depth_max)
        feat = self.pnp.features(frame)
        i = len(self.poses)
        if i == 0:
            T = np.eye(4)
            self.keyframes.append(dict(idx=0, rgbd=rgbd, T=T, feat=feat, depth=frame["depth"]))
            self.graph.nodes.append(reg.PoseGraphNode(T))
            self.rel.append((0, np.eye(4))); self.poses.append(T); self.sigmas.append(0.0); self.step_sigmas.append(0.0)
            self.timing.append(time.perf_counter() - t0)
            return T
        kf = self.keyframes[-1]
        # constant-velocity prediction expressed relative to the keyframe
        T_pred = self.poses[-1] if i < 2 else self.poses[-1] @ np.linalg.inv(self.poses[-2]) @ self.poses[-1]
        init_cv = np.linalg.inv(T_pred) @ kf["T"]                   # constant-velocity prior (kf -> current)
        best, best_score, info = None, -1.0, None

        def consider(init):
            nonlocal best, best_score, info
            ok, T_try, inf = self._odom(kf["rgbd"], rgbd, init)
            for T_c, inf_c in (((T_try, inf), (init, None)) if ok else ((init, None),)):
                sc = depth_consistency(kf["depth"], frame["depth"], self.K, T_c, depth_max=self.check_max)
                if sc > best_score:
                    best, best_score, info = T_c, sc, inf_c

        consider(init_cv)
        if self.use_pnp and (best_score < 0.75 or info is None):          # weak: try the feature-based (ORB + PnP) initialisation
            T_pnp, n_in = self.pnp.relative(kf["feat"], feat)
            if T_pnp is not None and n_in >= 15:
                consider(T_pnp)
        T_cur_kf = best
        if info is None or best_score < 0.35:
            self.n_track_fail += 1
            info = np.eye(6) * max(best_score, 0.05) * 1e3
        T_kf_cur = np.linalg.inv(T_cur_kf)
        T = kf["T"] @ T_kf_cur
        step = min(pose_sigma(info), 0.5)
        sig = self.sigmas[kf["idx"]] + step                  # accumulated (dead-reckoning) uncertainty
        self.rel.append((len(self.keyframes) - 1, T_kf_cur)); self.poses.append(T); self.sigmas.append(sig)
        self.step_sigmas.append(step)
        if np.linalg.norm(T_kf_cur[:3, 3]) > self.kf_trans or _rot_angle(T_kf_cur) > self.kf_rot:
            self._add_keyframe(i, rgbd, T, T_kf_cur, info, feat, frame["depth"])
        self.timing.append(time.perf_counter() - t0)
        return T

    def _add_keyframe(self, i, rgbd, T, T_prev_cur, info, feat, depth):
        k = len(self.keyframes)
        self.keyframes.append(dict(idx=i, rgbd=rgbd, T=T, feat=feat, depth=depth))
        self.graph.nodes.append(reg.PoseGraphNode(T))
        # Open3D edge convention: transformation maps source-frame points into the target frame
        self.graph.edges.append(reg.PoseGraphEdge(k - 1, k, np.linalg.inv(T_prev_cur), info, uncertain=False))
        self.rel[i] = (k, np.eye(4))
        if not self.use_loops:
            return
        # loop closure against older nearby keyframes
        c = T[:3, 3]
        for j, kj in enumerate(self.keyframes[:-self.loop_min_gap]):
            if np.linalg.norm(kj["T"][:3, 3] - c) > self.loop_radius:
                continue
            init = np.linalg.inv(T) @ kj["T"]
            T_pnp, n_in = self.pnp.relative(kj["feat"], feat)
            if T_pnp is None or n_in < 40:
                continue
            ok, T_cur_j, inf = self._odom(kj["rgbd"], rgbd, T_pnp)
            if not ok or depth_consistency(kj["depth"], depth, self.K, T_cur_j, depth_max=self.check_max) < 0.5:
                continue
            if np.linalg.norm((init @ np.linalg.inv(T_cur_j))[:3, 3]) > self.loop_radius + 0.5:   # implausible
                continue
            self.graph.edges.append(reg.PoseGraphEdge(j, k, T_cur_j, inf, uncertain=True))
            self.n_loops += 1

    def optimize(self):
        self.poses_before_opt = [T.copy() for T in self.poses]
        if self.n_loops == 0:
            return self.poses
        option = reg.GlobalOptimizationOption(max_correspondence_distance=0.05,
                                              edge_prune_threshold=0.25, reference_node=0)
        reg.global_optimization(self.graph, reg.GlobalOptimizationLevenbergMarquardt(),
                                reg.GlobalOptimizationConvergenceCriteria(), option)
        for k, node in enumerate(self.graph.nodes):
            self.keyframes[k]["T"] = np.asarray(node.pose)
        self.poses = [self.keyframes[k]["T"] @ Trel for (k, Trel) in self.rel]
        return self.poses


def run_slam(seq, depth_max=6.0, verbose=True, **kw):
    f0 = seq.frame(0)
    h, w = f0["depth"].shape
    slam = RGBDSLAM(seq.K, w, h, depth_max=depth_max, **kw)
    for i in range(len(seq)):
        slam.track(seq.frame(i))
        if verbose and i % 50 == 0:
            print(f"  tracked {i}/{len(seq)}  keyframes={len(slam.keyframes)}  loops={slam.n_loops}")
    t0 = time.perf_counter()
    last = len(slam.poses) - 1
    if slam.rel[last][0] != len(slam.keyframes) - 1 or not np.allclose(slam.rel[last][1], np.eye(4)):
        f = seq.frame(last)                                  # close the trajectory with a final keyframe
        k, T_k_cur = slam.rel[last]
        slam._add_keyframe(last, make_rgbd(f, slam.depth_max), slam.poses[last], T_k_cur, np.eye(6) * 1e3,
                           slam.pnp.features(f), f["depth"])
    poses = slam.optimize()
    slam.t_opt = time.perf_counter() - t0
    return slam, poses


def load_trajectory(path, timestamps, max_dt=0.03):
    """Load a TUM-format trajectory (e.g. VIGS-SLAM traj_*.txt) and associate it to frames."""
    from .data import quat_to_R
    rows = np.loadtxt(path)
    out = []
    for t in timestamps:
        j = int(np.argmin(np.abs(rows[:, 0] - t)))
        tx, ty, tz, qx, qy, qz, qw = rows[j, 1:8]
        T = np.eye(4); T[:3, :3] = quat_to_R(qw, qx, qy, qz); T[:3, 3] = (tx, ty, tz)
        out.append(T if abs(rows[j, 0] - t) < max_dt else None)
    return out

In [ ]:
%%writefile /content/geosemtwinx/metrics.py
"""Trajectory metrics (TUM-benchmark style)."""
import numpy as np


def umeyama(src, dst, with_scale=False):
    """Find s,R,t minimising ||dst - (s R src + t)||."""
    mu_s, mu_d = src.mean(0), dst.mean(0)
    xs, xd = src - mu_s, dst - mu_d
    U, D, Vt = np.linalg.svd(xd.T @ xs / len(src))
    S = np.eye(3)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0:
        S[2, 2] = -1
    R = U @ S @ Vt
    s = (D * np.diag(S)).sum() / xs.var(0).sum() if with_scale else 1.0
    t = mu_d - s * R @ mu_s
    return s, R, t


def ate(est, gt, with_scale=False):
    """Absolute trajectory error after SE(3)/Sim(3) alignment. Returns dict + aligned positions."""
    pe = np.array([T[:3, 3] for T in est]); pg = np.array([T[:3, 3] for T in gt])
    s, R, t = umeyama(pe, pg, with_scale)
    al = (s * (R @ pe.T)).T + t
    err = np.linalg.norm(al - pg, axis=1)
    T_align = np.eye(4); T_align[:3, :3] = s * R; T_align[:3, 3] = t
    return dict(rmse=float(np.sqrt((err ** 2).mean())), mean=float(err.mean()),
                median=float(np.median(err)), max=float(err.max()), T_align=T_align,
                errors=err, aligned=al, gt=pg)


def rpe(est, gt, timestamps, delta_s=1.0):
    """Relative pose error over a fixed time interval (default 1 s)."""
    ts = np.asarray(timestamps)
    terr, rerr = [], []
    for i in range(len(est)):
        j = int(np.searchsorted(ts, ts[i] + delta_s))
        if j >= len(est):
            break
        E = np.linalg.inv(np.linalg.inv(gt[i]) @ gt[j]) @ (np.linalg.inv(est[i]) @ est[j])
        terr.append(np.linalg.norm(E[:3, 3]))
        rerr.append(np.degrees(np.arccos(np.clip((np.trace(E[:3, :3]) - 1) / 2, -1, 1))))
    terr, rerr = np.array(terr), np.array(rerr)
    return dict(trans_rmse=float(np.sqrt((terr ** 2).mean())), rot_rmse_deg=float(np.sqrt((rerr ** 2).mean())),
                n=len(terr))


def path_length(poses):
    p = np.array([T[:3, 3] for T in poses])
    return float(np.linalg.norm(np.diff(p, axis=0), axis=1).sum())

In [ ]:
%%writefile /content/geosemtwinx/mapping.py
"""Dense mapping: uncertainty-weighted fused surface map, log-odds occupancy + uncertainty, traversability."""
import numpy as np
import open3d as o3d



def depth_sigma(z):
    """Structured-light / ToF style axial noise model (metres)."""
    return 0.0012 + 0.0019 * (np.maximum(z, 0.4) - 0.4) ** 2


def backproject(depth, K, T_wc, stride=1, mask=None, max_depth=None):
    h, w = depth.shape
    v, u = np.mgrid[0:h:stride, 0:w:stride]
    z = depth[::stride, ::stride]
    m = z > 0
    if max_depth:
        m &= z < max_depth
    if mask is not None:
        m &= mask[::stride, ::stride]
    x = (u[m] - K[0, 2]) / K[0, 0] * z[m]
    y = (v[m] - K[1, 2]) / K[1, 1] * z[m]
    P = np.stack([x, y, z[m]], 1)
    return P @ T_wc[:3, :3].T + T_wc[:3, 3], z[m], (v[m], u[m])


class FusedMap:
    """Uncertainty-weighted voxel fusion map (surfel-style), pure NumPy.

    Every depth pixel is back-projected and accumulated into a voxel hash with weight
    w = 1/sigma_z(z)^2 * exp(-sigma_pose / 0.1): far, noisy depth and badly-tracked frames count less.
    Frames whose pose sigma exceeds a gate are skipped and depth is cut where the sensor noise exceeds
    sigma_max. Each voxel keeps the weighted mean position and colour, the total weight and the number
    of observations; voxels seen fewer than `min_obs` times are dropped as noise when exporting.
    (Replaces Open3D's ScalableTSDFVolume, which returns empty surfaces in the Open3D 0.20 / Python 3.13
    build used on Colab.)"""

    def __init__(self, K, width, height, voxel=0.02, depth_max=4.0, sigma_max=0.03, pose_sigma_gate=0.15,
                 stride=None, min_obs=2):
        self.K = np.asarray(K, float)
        self.voxel = float(voxel)
        zs = np.linspace(0.4, 25, 4000)
        self.depth_max = float(min(depth_max, zs[depth_sigma(zs) < sigma_max].max()))
        self.gate = pose_sigma_gate
        # sample roughly 2 pixels per voxel footprint at mid range
        self.stride = stride or max(1, int(K[0, 0] * voxel / max(self.depth_max / 2, 0.5) / 2))
        self.min_obs = min_obs
        self.n_integrated = self.n_skipped = 0
        self._buf = []
        self.keys = np.empty(0, np.int64); self.acc = np.empty((0, 8))      # xyz*w, rgb*w, w, count

    def integrate(self, frame, T_wc, pose_sigma=0.0):
        if pose_sigma > self.gate:
            self.n_skipped += 1
            return
        P, z, (v, u) = backproject(frame["depth"], self.K, T_wc, stride=self.stride, max_depth=self.depth_max)
        if len(P) == 0:
            return
        w = (1.0 / depth_sigma(z) ** 2) * np.exp(-pose_sigma / 0.1)
        w = w / w.max()
        col = frame["rgb"][v, u].astype(np.float64) / 255.0
        ijk = np.floor(P / self.voxel).astype(np.int64) + (1 << 20)
        k = (ijk[:, 0] << 42) | (ijk[:, 1] << 21) | ijk[:, 2]
        self._buf.append((k, np.column_stack([P * w[:, None], col * w[:, None], w, np.ones_like(w)])))
        self.n_integrated += 1
        if sum(len(b[0]) for b in self._buf) > 1_500_000:
            self._consolidate()

    def _consolidate(self):
        if not self._buf:
            return
        keys = np.concatenate([self.keys] + [b[0] for b in self._buf])
        acc = np.concatenate([self.acc] + [b[1] for b in self._buf])
        self._buf = []
        u, inv = np.unique(keys, return_inverse=True)
        out = np.zeros((len(u), acc.shape[1]))
        np.add.at(out, inv, acc)
        self.keys, self.acc = u, out

    def arrays(self, min_obs=None):
        self._consolidate()
        m = self.acc[:, 7] >= (min_obs or self.min_obs)
        a = self.acc[m]
        w = a[:, 6:7]
        return a[:, 0:3] / w, np.clip(a[:, 3:6] / w, 0, 1), a[:, 6], a[:, 7].astype(int)

    def point_cloud(self):
        P, C, _, _ = self.arrays()
        pc = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(P))
        pc.colors = o3d.utility.Vector3dVector(C)
        return pc


TSDFMap = FusedMap          # backwards-compatible name


class VoxelGrid:
    """Dense voxel grid helper (index <-> world)."""

    def __init__(self, bounds_min, bounds_max, res):
        self.res = float(res)
        self.min = np.asarray(bounds_min, float)
        self.shape = np.ceil((np.asarray(bounds_max, float) - self.min) / res).astype(int)

    def index(self, P):
        ijk = np.floor((P - self.min) / self.res).astype(np.int64)
        ok = np.all((ijk >= 0) & (ijk < self.shape), axis=1)
        return ijk, ok

    def flat(self, ijk):
        return (ijk[:, 0] * self.shape[1] + ijk[:, 1]) * self.shape[2] + ijk[:, 2]

    def center(self, ijk):
        return self.min + (ijk + 0.5) * self.res


class OccupancyMap(VoxelGrid):
    """Log-odds occupancy + per-voxel reconstruction uncertainty (fused depth sigma)."""
    L_OCC, L_FREE, L_MIN, L_MAX = 0.85, -0.4, -2.0, 3.5

    def __init__(self, bounds_min, bounds_max, res=0.1, max_range=6.0):
        super().__init__(bounds_min, bounds_max, res)
        n = int(np.prod(self.shape))
        self.logodds = np.zeros(n, np.float32)
        self.inv_var = np.zeros(n, np.float32)      # sum of 1/sigma^2 of surface observations
        self.hits = np.zeros(n, np.uint16)
        self.max_range = max_range

    def integrate(self, depth, K, T_wc, stride=8, pose_sigma=0.0):
        P, z, _ = backproject(depth, K, T_wc, stride=stride, max_depth=self.max_range)
        self.integrate_points(P, T_wc[:3, 3], z, pose_sigma)

    def integrate_points(self, P, origin, z=None, pose_sigma=0.0):
        if len(P) == 0:
            return
        d = P - origin
        L = np.linalg.norm(d, axis=1)
        steps = np.arange(0.25, self.max_range, self.res * 0.9)
        # free space samples along each ray, stopping one voxel before the hit
        frac = steps[None, :] / np.maximum(L[:, None], 1e-6)
        keep = steps[None, :] < (L[:, None] - self.res)
        S = origin + d[:, None, :] * frac[..., None]
        S = S[keep]
        ijk, ok = self.index(S)
        free = np.unique(self.flat(ijk[ok]))
        ijk_h, ok_h = self.index(P)
        hit = self.flat(ijk_h[ok_h])
        hit_u, cnt = np.unique(hit, return_counts=True)
        free = np.setdiff1d(free, hit_u, assume_unique=True)
        self.logodds[free] = np.clip(self.logodds[free] + self.L_FREE, self.L_MIN, self.L_MAX)
        self.logodds[hit_u] = np.clip(self.logodds[hit_u] + self.L_OCC, self.L_MIN, self.L_MAX)
        sig = depth_sigma(z if z is not None else L)[ok_h] + pose_sigma
        np.add.at(self.inv_var, hit, (1.0 / np.maximum(sig, 1e-3) ** 2).astype(np.float32))
        self.hits[hit_u] = np.minimum(self.hits[hit_u].astype(np.int32) + cnt, 65000).astype(np.uint16)

    # states: -1 unknown, 0 free, 1 occupied
    def state(self):
        s = np.full(self.logodds.shape, -1, np.int8)
        s[self.logodds < -0.3] = 0
        s[self.logodds > 0.6] = 1
        return s

    def surface_sigma(self):
        with np.errstate(divide="ignore"):
            return np.where(self.inv_var > 0, 1.0 / np.sqrt(self.inv_var), np.inf)

    def coverage(self, gt_surface_flat):
        st = self.state()
        return float((st[gt_surface_flat] == 1).mean())

    def traversability(self, floor_z=0.0, robot_height=1.8, clearance=0.15):
        """2D grid: 1 traversable, 0 obstacle, -1 unknown."""
        st = self.state().reshape(self.shape)
        kz0 = int(np.floor((floor_z + clearance - self.min[2]) / self.res))
        kz1 = int(np.ceil((floor_z + robot_height - self.min[2]) / self.res))
        kz0, kz1 = max(kz0, 0), min(kz1, self.shape[2])
        band = st[:, :, kz0:kz1]
        occ = (band == 1).any(axis=2)
        known_free = (band == 0).sum(axis=2) >= 1
        grid = np.full(occ.shape, -1, np.int8)
        grid[known_free] = 1
        grid[occ] = 0
        return grid

In [ ]:
%%writefile /content/geosemtwinx/gasf.py
"""Geometry-Aware Semantic Fusion (GASF) and baselines.

A 2D detection (label, score, mask) is lifted to a 3D *segment* with the
estimated camera pose. Segments are associated to persistent 3D *instances*
with a score that combines

    semantic agreement      (fused class distribution, cosine)
    voxel overlap           (3D support overlap, 0.2 m voxels)
    centroid proximity      (size-adaptive Gaussian)
    geometric descriptor    (log extents, PCA shape, surface-normal histogram)

Matching is one-to-one (Hungarian). Labels are fused with confidence weights
(detector score x mask score x depth quality x pose confidence). Instances are
confirmed only after multi-view agreement, stale false positives are removed by
visibility-checked misses, and duplicates are merged by 3D overlap.
"""
import numpy as np
import open3d as o3d
from scipy import ndimage
from scipy.optimize import linear_sum_assignment

from .mapping import backproject, depth_sigma

OFF = 1 << 20


def vkeys(P, res):
    ijk = np.floor(P / res).astype(np.int64) + OFF
    return np.unique((ijk[:, 0] << 42) | (ijk[:, 1] << 21) | ijk[:, 2])


_NB = np.array([(a << 42) | (b << 21) | c for a in (-1, 0, 1) for b in (-1, 0, 1) for c in (-1, 0, 1)], np.int64)


def dilate_keys(keys):
    """26-neighbourhood dilation of encoded voxel keys."""
    return np.unique((keys[:, None] + _NB[None, :]).ravel())


def keys_to_centers(keys, res):
    i = (keys >> 42) - OFF; j = ((keys >> 21) & (2 ** 21 - 1)) - OFF; k = (keys & (2 ** 21 - 1)) - OFF
    return (np.stack([i, j, k], 1) + 0.5) * res


def largest_near_component(P, z, res=0.1):
    """Voxel connected components (26-nbhd); keep the biggest component that is not
    behind the object (rejects mask bleed onto background)."""
    if len(P) < 30:
        return np.ones(len(P), bool)
    ijk = np.floor((P - P.min(0)) / res).astype(np.int64)
    shape = ijk.max(0) + 1
    if np.prod(shape) > 4e6:                                   # pathological mask, fall back to depth gate
        return z < np.median(z) + 0.5
    grid = np.zeros(shape, bool); grid[tuple(ijk.T)] = True
    lab, n = ndimage.label(grid, structure=np.ones((3, 3, 3)))
    pl = lab[tuple(ijk.T)]
    zmed = np.percentile(z, 40)
    best, best_n = 1, -1
    for c in range(1, n + 1):
        m = pl == c
        cnt = m.sum()
        if cnt > best_n and np.median(z[m]) <= zmed + 0.6:
            best, best_n = c, cnt
    return pl == best


def descriptor(P):
    ext = np.sort(np.ptp(P, 0))[::-1] + 1e-3
    C = np.cov((P - P.mean(0)).T) + 1e-9 * np.eye(3)
    ev = np.sort(np.linalg.eigvalsh(C))[::-1]
    l1, l2, l3 = ev / ev.sum()
    pc = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(P[:: max(1, len(P) // 1500)]))
    pc.estimate_normals(o3d.geometry.KDTreeSearchParamKNN(16))
    nz = np.abs(np.asarray(pc.normals)[:, 2])
    hist = np.histogram(nz, bins=4, range=(0, 1))[0] / max(len(nz), 1)
    return np.concatenate([np.log(ext), [(l1 - l2) / l1, (l2 - l3) / l1, l3 / l1], 2 * hist])


class Segment:
    __slots__ = ("sid", "frame", "label", "score", "probs", "pts", "centroid", "extent", "vox",
                 "desc", "weight", "cam", "gt", "gt_purity", "npix")


def make_segments(frame, det_list, K, T_wc, classes, frame_idx, pose_sigma=0.0, max_depth=8.0,
                  gt_instance=None, start_sid=0, min_points=60, floor_classes=(), floor_z=0.0, floor_gap=0.6,
                  min_score=0.0):
    """floor_classes: classes that must touch the floor (e.g. forklift, pallet truck). A detection of
    such a class whose 3D support starts > floor_gap above the floor is rejected (geometric sanity check)."""
    cls_index = {c: i for i, c in enumerate(classes)}
    segs = []
    for d in det_list:
        if d["label"] not in cls_index or d["score"] < min_score:
            continue
        P, z, (vv, uu) = backproject(frame["depth"], K, T_wc, mask=d["mask"], max_depth=max_depth)
        if len(P) < min_points:
            continue
        keep = largest_near_component(P, z)
        P, z, vv, uu = P[keep], z[keep], vv[keep], uu[keep]
        if len(P) < min_points:
            continue
        if d["label"] in floor_classes and np.percentile(P[:, 2], 2) > floor_z + floor_gap:
            continue
        s = Segment()
        s.sid, s.frame, s.label, s.score = start_sid + len(segs), frame_idx, d["label"], float(d["score"])
        s.probs = np.zeros(len(classes)); s.probs[cls_index[d["label"]]] = 1.0
        sub = P[:: max(1, len(P) // 1500)]
        s.pts = sub.astype(np.float32)
        s.centroid = P.mean(0); s.extent = np.ptp(P, 0)
        s.vox = vkeys(P, 0.2)
        s.desc = descriptor(sub)
        q_depth = float(np.exp(-np.median(z) / 6.0))
        q_pose = float(np.exp(-pose_sigma / 0.25))
        s.weight = s.score * d.get("mask_score", 1.0) * q_depth * q_pose
        s.cam = T_wc[:3, 3].copy()
        s.npix = int(d["mask"].sum())
        s.gt, s.gt_purity = 0, 0.0
        if gt_instance is not None:
            ids = gt_instance[vv, uu]
            vals, cnt = np.unique(ids, return_counts=True)
            j = int(np.argmax(cnt))
            s.gt, s.gt_purity = int(vals[j]), float(cnt[j] / cnt.sum())
            if s.gt_purity < 0.5:
                s.gt = 0
        segs.append(s)
    return segs


class Instance:
    def __init__(self, iid, seg, n_cls):
        self.iid = iid
        self.scores = np.zeros(n_cls)
        self.vox = np.empty(0, np.int64)
        self.pts = np.empty((0, 3), np.float32)
        self.desc = None
        self.wsum = 0.0
        self.centroid = None
        self.views, self.segs = [], []
        self.hits, self.misses = 0, 0
        self.first, self.last = seg.frame, seg.frame
        self.add(seg)

    def add(self, s):
        w = max(s.weight, 1e-3)
        self.scores += w * s.probs
        self.centroid = s.centroid if self.centroid is None else (self.wsum * self.centroid + w * s.centroid) / (self.wsum + w)
        self.desc = s.desc if self.desc is None else (self.wsum * self.desc + w * s.desc) / (self.wsum + w)
        self.wsum += w
        self.vox = np.union1d(self.vox, s.vox)
        P = np.vstack([self.pts, s.pts])
        if len(P) > 6000:
            ijk = np.floor(P / 0.04).astype(np.int64)
            _, idx = np.unique(ijk, axis=0, return_index=True)
            P = P[idx]
        self.pts = P
        if all(np.linalg.norm(s.cam - v) > 0.25 for v in self.views):
            self.views.append(s.cam)
        self.segs.append(s)
        self.hits += 1
        self.last = s.frame

    @property
    def probs(self):
        return self.scores / max(self.scores.sum(), 1e-9)

    @property
    def extent(self):
        """Robust size (2nd-98th percentile) so a few stray points do not inflate the asset."""
        if len(self.pts) < 5:
            return np.ptp(self.pts, 0) if len(self.pts) else np.zeros(3)
        lo, hi = np.percentile(self.pts, [2, 98], axis=0)
        return hi - lo

    def robust_bounds(self, q=2):
        lo, hi = np.percentile(self.pts, [q, 100 - q], axis=0)
        return lo, hi

    def label_entropy(self):
        p = self.probs[self.probs > 0]
        return float(-(p * np.log(p)).sum())

    def existence(self):
        return self.hits / (self.hits + self.misses)


class GASF:
    def __init__(self, classes, w_sem=0.25, w_ov=0.35, w_cen=0.2, w_geo=0.2, accept=0.45,
                 confirm_obs=2, confirm_views=2, use_geometry=True, use_multiview=True, merge=True):
        self.classes = list(classes)
        self.w = np.array([w_sem, w_ov, w_cen, w_geo], float)
        if not use_geometry:
            self.w = np.array([0.5, 0.0, 0.5, 0.0])
        self.accept, self.confirm_obs, self.confirm_views = accept, confirm_obs, confirm_views
        self.use_geometry, self.use_multiview, self.do_merge = use_geometry, use_multiview, merge
        self.instances = {}
        self.next_id = 1
        self.assign = {}            # segment id -> instance id (final, after merges)
        self.n_frames = 0

    def _score(self, s, inst):
        sem = float(s.probs @ inst.probs / (np.linalg.norm(inst.probs) + 1e-9))
        inter = np.intersect1d(s.vox, inst.vox, assume_unique=True).size
        ov = inter / max(min(s.vox.size, inst.vox.size), 1)
        size = max(s.extent.max(), inst.extent.max())
        sig = 0.25 + 0.25 * size
        d = np.linalg.norm(s.centroid - inst.centroid)
        cen = float(np.exp(-d * d / (2 * sig * sig)))
        geo = float(np.exp(-np.abs(s.desc - inst.desc).sum() / 2.0))
        S = float(self.w @ np.array([sem, ov, cen, geo]))
        if self.use_geometry:
            # strong 3D overlap with a similar shape overrides a (possibly wrong) 2D label
            ok = (sem >= 0.2 and S >= self.accept and (ov >= 0.1 or cen >= 0.5)) or (ov >= 0.5 and geo >= 0.5)
        else:
            ok = sem >= 0.2 and S >= self.accept
        return S, ok

    def update(self, segs, frame=None, K=None, T_wc=None):
        self.n_frames += 1
        active = [i for i in self.instances.values()]
        if segs and active:
            cost = np.full((len(segs), len(active)), 10.0)
            for a, s in enumerate(segs):
                for b, inst in enumerate(active):
                    if np.linalg.norm(s.centroid - inst.centroid) > 3.0 + s.extent.max() + inst.extent.max():
                        continue
                    S, ok = self._score(s, inst)
                    if ok:
                        cost[a, b] = 1.0 - S
            rows, cols = linear_sum_assignment(cost)
            matched = {r: c for r, c in zip(rows, cols) if cost[r, c] < 10.0}
        else:
            matched = {}
        touched = set()
        for a, s in enumerate(segs):
            if a in matched:
                inst = active[matched[a]]
                inst.add(s)
            else:
                inst = Instance(self.next_id, s, len(self.classes))
                self.instances[self.next_id] = inst
                self.next_id += 1
            self.assign[s.sid] = inst.iid
            touched.add(inst.iid)
        if frame is not None and K is not None:
            self._visibility_misses(frame, K, T_wc, touched)
        if self.do_merge and self.n_frames % 5 == 0:
            self.merge_duplicates()

    def _visibility_misses(self, frame, K, T_wc, touched):
        """An instance that should be visible (in FOV, not occluded, close) but was not
        detected accumulates a miss; tentative ones with low existence are pruned."""
        h, w = frame["depth"].shape
        Tcw = np.linalg.inv(T_wc)
        for iid in list(self.instances):
            if iid in touched:
                continue
            inst = self.instances[iid]
            pc = Tcw[:3, :3] @ inst.centroid + Tcw[:3, 3]
            if pc[2] < 0.5 or pc[2] > 6.0:
                continue
            u = K[0, 0] * pc[0] / pc[2] + K[0, 2]; v = K[1, 1] * pc[1] / pc[2] + K[1, 2]
            if not (0.1 * w < u < 0.9 * w and 0.1 * h < v < 0.9 * h):
                continue
            dz = frame["depth"][int(v), int(u)]
            if dz > 0 and dz < pc[2] - 0.4:                       # occluded
                continue
            inst.misses += 1
            if not self.is_confirmed(inst) and inst.misses >= 3 and inst.existence() < 0.4:
                del self.instances[iid]
                for s in inst.segs:
                    self.assign.pop(s.sid, None)

    def _should_merge(self, A, B):
        if np.linalg.norm(A.centroid - B.centroid) > 1.5 + max(A.extent.max(), B.extent.max()):
            return False
        inter = np.intersect1d(A.vox, B.vox, assume_unique=True).size
        iou = inter / max(np.union1d(A.vox, B.vox).size, 1)
        small = inter / max(min(A.vox.size, B.vox.size), 1)
        same = np.argmax(A.probs) == np.argmax(B.probs)
        if (same and (iou > 0.3 or small > 0.6)) or iou > 0.55:
            return True
        if same and self.use_geometry:
            # fragments of one object (split masks, partial views): they touch, and the union is
            # no larger than the bigger fragment's extent allows (rejects two adjacent objects)
            Bd = dilate_keys(B.vox) if B.vox.size < A.vox.size else B.vox
            Ad = A.vox if B.vox.size < A.vox.size else dilate_keys(A.vox)
            touch = np.intersect1d(Ad, Bd).size / max(min(A.vox.size, B.vox.size), 1)
            P = np.vstack([A.pts, B.pts])
            lo, hi = np.percentile(P, [2, 98], axis=0)
            union_ext = (hi - lo).max()
            if touch > 0.4 and union_ext <= 1.25 * max(A.extent.max(), B.extent.max()):
                return True
        return False

    def merge_duplicates(self):
        merged = True
        while merged:
            merged = False
            ids = sorted(self.instances)
            for a in range(len(ids)):
                for b in range(a + 1, len(ids)):
                    A, B = self.instances[ids[a]], self.instances[ids[b]]
                    if self._should_merge(A, B):
                        keep, drop = (A, B) if A.wsum >= B.wsum else (B, A)
                        for s in drop.segs:
                            keep.add(s); self.assign[s.sid] = keep.iid
                        keep.hits -= len(drop.segs); keep.hits += drop.hits; keep.misses += drop.misses
                        del self.instances[drop.iid]
                        merged = True
                        break
                if merged:
                    break

    def is_confirmed(self, inst):
        if not self.use_multiview:
            return True
        return inst.hits >= self.confirm_obs and len(inst.views) >= self.confirm_views and inst.existence() >= 0.5

    def finalize(self):
        if self.do_merge:
            self.merge_duplicates()
        return [i for i in self.instances.values() if self.is_confirmed(i)]

    def objects(self):
        """Uniform output for evaluation: list of (label, [segments])."""
        return [(self.classes[int(np.argmax(i.probs))], i.segs, i) for i in self.finalize()]


# ----------------------------------------------------------------------------- baselines
def baseline_independent(all_segs, classes):
    """Independent 2D labelling: every detection becomes its own object."""
    return [(s.label, [s], None) for s in all_segs]


def baseline_centroid(all_segs, classes, radius=1.0, min_obs=2):
    """Greedy same-label nearest-centroid tracking (no 3D overlap / shape terms). Tracks seen fewer than
    min_obs times are dropped (same observation filter GASF uses, for a fair comparison)."""
    objs = []
    for s in sorted(all_segs, key=lambda x: x.frame):
        best, bd = None, radius
        for o in objs:
            if o["label"] != s.label:
                continue
            d = np.linalg.norm(o["c"] - s.centroid)
            if d < bd:
                best, bd = o, d
        if best is None:
            objs.append(dict(label=s.label, c=s.centroid.copy(), segs=[s]))
        else:
            n = len(best["segs"]); best["c"] = (best["c"] * n + s.centroid) / (n + 1); best["segs"].append(s)
    return [(o["label"], o["segs"], None) for o in objs if len(o["segs"]) >= min_obs]


def baseline_geometry_clustering(all_segs, classes, res=0.1, min_obs=2):
    """Geometry-only: connected components over all lifted points, majority label; clusters supported by
    fewer than min_obs detections are dropped."""
    if not all_segs:
        return []
    P = np.vstack([s.pts for s in all_segs])
    owner = np.concatenate([np.full(len(s.pts), k) for k, s in enumerate(all_segs)])
    ijk = np.floor(P / res).astype(np.int64)
    mn = ijk.min(0); ijk -= mn
    shape = ijk.max(0) + 1
    if np.prod(shape) > 5e7:
        res *= 2; ijk = np.floor(P / res).astype(np.int64); ijk -= ijk.min(0); shape = ijk.max(0) + 1
    grid = np.zeros(shape, bool); grid[tuple(ijk.T)] = True
    lab, n = ndimage.label(grid, structure=np.ones((3, 3, 3)))
    pl = lab[tuple(ijk.T)]
    objs = []
    for c in range(1, n + 1):
        m = pl == c
        members = np.unique(owner[m])
        segs = [all_segs[k] for k in members]
        if len(segs) < min_obs:
            continue
        labels = [s.label for s in segs]
        lab_major = max(set(labels), key=labels.count)
        objs.append((lab_major, segs, None))
    return objs

In [ ]:
%%writefile /content/geosemtwinx/detectors.py
"""2D perception front ends.

GroundedSAM2Detector  - open-vocabulary detection (Grounding DINO) + promptable
                        segmentation (SAM 2.1). Needs a GPU runtime (Colab T4 is fine).
OracleNoisyDetector   - derives detections from ground-truth instance masks and
                        injects realistic detector errors (misses, label confusion,
                        mask bleed, split masks, false positives). Used to evaluate
                        the fusion stage in isolation with exact ground truth.
"""
import numpy as np
from scipy import ndimage


def box_iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1]); x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter + 1e-9)


class GroundedSAM2Detector:
    def __init__(self, classes, device="cuda", dino_id="IDEA-Research/grounding-dino-tiny",
                 sam_cfg="configs/sam2.1/sam2.1_hiera_t.yaml", sam_ckpt="checkpoints/sam2.1_hiera_tiny.pt",
                 box_threshold=0.30, text_threshold=0.25, max_box_frac=0.5, nms_iou=0.7):
        import torch
        from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor
        from sam2.build_sam import build_sam2
        from sam2.sam2_image_predictor import SAM2ImagePredictor
        self.torch = torch
        self.classes = list(classes)
        self.prompt = " ".join(c + "." for c in self.classes)
        self.proc = AutoProcessor.from_pretrained(dino_id)
        self.dino = AutoModelForZeroShotObjectDetection.from_pretrained(dino_id).to(device).eval()
        self.sam = SAM2ImagePredictor(build_sam2(sam_cfg, sam_ckpt, device=device))
        self.device = device
        self.bt, self.tt, self.max_frac, self.nms = box_threshold, text_threshold, max_box_frac, nms_iou

    def _cls(self, text):
        text = text.lower()
        best = None
        for c in sorted(self.classes, key=len, reverse=True):     # longest match first
            if c in text or text in c:
                best = c; break
        if best is None:
            for c in self.classes:
                if any(w in text for w in c.split()):
                    best = c; break
        return best

    def __call__(self, rgb):
        from PIL import Image
        torch = self.torch
        H, W = rgb.shape[:2]
        img = Image.fromarray(rgb)
        inputs = self.proc(images=img, text=self.prompt, return_tensors="pt").to(self.device)
        with torch.no_grad():
            out = self.dino(**inputs)
        res = self.proc.post_process_grounded_object_detection(
            out, input_ids=inputs["input_ids"], threshold=self.bt, text_threshold=self.tt,
            target_sizes=[(H, W)], text_labels=[self.classes])[0]
        labels = res.get("text_labels", res.get("labels"))
        dets = []
        for box, score, lab in zip(res["boxes"].cpu().numpy(), res["scores"].cpu().numpy(), labels):
            c = self._cls(str(lab))
            if c is None:
                continue
            if (box[2] - box[0]) * (box[3] - box[1]) > self.max_frac * W * H:
                continue
            dets.append(dict(label=c, score=float(score), box=box))
        dets.sort(key=lambda d: -d["score"])
        kept = []
        for d in dets:
            if all(box_iou(d["box"], k["box"]) < self.nms for k in kept):
                kept.append(d)
        if not kept:
            return []
        self.sam.set_image(rgb)
        for k in kept:
            masks, mscores, _ = self.sam.predict(box=k["box"].astype(np.float32), multimask_output=False)
            k["mask"] = np.asarray(masks[0]).astype(bool)
            k["mask_score"] = float(np.asarray(mscores).ravel()[0])
        return kept


class OracleNoisyDetector:
    """Ground-truth masks + error model calibrated to typical open-vocabulary detector behaviour."""

    def __init__(self, classes, instance_class, seed=0, p_miss=0.15, p_miss_small=0.6, min_area=350,
                 p_confuse=0.08, p_bleed=0.15, p_split=0.06, p_dup=0.05, p_fp=0.12, confusions=None):
        self.classes = list(classes)
        self.instance_class = instance_class
        self.rng = np.random.default_rng(seed)
        self.p = dict(miss=p_miss, miss_small=p_miss_small, confuse=p_confuse, bleed=p_bleed,
                      split=p_split, dup=p_dup, fp=p_fp)
        self.min_area = min_area
        self.conf = confusions or {"pallet": "cardboard box", "cardboard box": "pallet",
                                   "barrel": "cardboard box", "forklift": "pallet truck"}

    def __call__(self, frame):
        rng, p = self.rng, self.p
        inst = frame["instance"]
        H, W = inst.shape
        dets = []
        ids, cnt = np.unique(inst, return_counts=True)
        for iid, area in zip(ids, cnt):
            if iid == 0 or iid not in self.instance_class:
                continue
            cls = self.instance_class[iid]
            if cls not in self.classes:
                continue
            if rng.random() < (p["miss_small"] if area < self.min_area else p["miss"]):
                continue
            m = inst == iid
            label = cls
            if rng.random() < p["confuse"]:
                alt = self.conf.get(cls)
                if alt in self.classes:
                    label = alt
            r = rng.random()
            if r < p["bleed"]:
                m = ndimage.binary_dilation(m, iterations=int(rng.integers(6, 14)))
            elif r < p["bleed"] + 0.3:
                it = int(rng.integers(1, 3))
                m = ndimage.binary_erosion(m, iterations=it) if rng.random() < 0.5 else ndimage.binary_dilation(m, iterations=it)
            score = float(np.clip(rng.normal(0.48, 0.07), 0.3, 0.8))
            parts = [m]
            if rng.random() < p["split"] and area > 2000:
                ys, xs = np.nonzero(m)
                cut = np.median(xs)
                a = m.copy(); a[:, int(cut):] = False
                b = m.copy(); b[:, :int(cut)] = False
                parts = [a, b]
            for part in parts:
                if part.sum() < 50:
                    continue
                dets.append(dict(label=label, score=score, mask=part, mask_score=float(rng.uniform(0.85, 0.98))))
            if rng.random() < p["dup"]:
                dets.append(dict(label=label, score=score * 0.8, mask=ndimage.binary_erosion(m, iterations=2),
                                 mask_score=0.9))
        if rng.random() < p["fp"]:                     # hallucinated object on background
            cy, cx = int(rng.integers(H // 4, 3 * H // 4)), int(rng.integers(W // 6, 5 * W // 6))
            hh, ww = int(rng.integers(20, 70)), int(rng.integers(20, 90))
            m = np.zeros((H, W), bool); m[max(cy - hh, 0):cy + hh, max(cx - ww, 0):cx + ww] = True
            m &= frame["depth"] > 0
            if m.sum() > 200:
                dets.append(dict(label=self.classes[int(rng.integers(len(self.classes)))],
                                 score=float(rng.uniform(0.3, 0.42)), mask=m, mask_score=float(rng.uniform(0.6, 0.85))))
        return dets

In [ ]:
%%writefile /content/geosemtwinx/evaluation.py
"""Evaluation of semantic mapping / persistent identity against ground truth."""
from collections import Counter, defaultdict

import numpy as np

from .gasf import vkeys


def object_gt(segs):
    w = defaultdict(float)
    for s in segs:
        w[s.gt] += len(s.pts)
    g = max(w, key=w.get)
    purity = w[g] / sum(w.values())
    return g, purity


def evaluate_objects(objects, all_segs, instance_class, visible_gt):
    """objects: list of (label, segs, _). visible_gt: GT ids that were detectable.

    Returns precision/recall/F1 at asset level, duplicate rate, label accuracy,
    identity consistency (share of GT-labelled observations that end up in the
    single primary object of their true asset) and ID switches."""
    rows = []
    for k, (label, segs, _) in enumerate(objects):
        g, pur = object_gt(segs)
        rows.append(dict(k=k, label=label, gt=g, purity=pur, support=sum(len(s.pts) for s in segs), segs=segs))
    by_gt = defaultdict(list)
    for r in rows:
        if r["gt"] != 0 and r["purity"] >= 0.5:
            by_gt[r["gt"]].append(r)
    primary = {g: max(rs, key=lambda r: r["support"]) for g, rs in by_gt.items()}
    tp = len(primary)
    dup = sum(len(rs) - 1 for rs in by_gt.values())
    fp = len(rows) - tp - dup
    n_gt = len(visible_gt)
    prec = tp / max(len(rows), 1)
    rec = len(set(primary) & set(visible_gt)) / max(n_gt, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-9)
    lab_acc = np.mean([primary[g]["label"] == instance_class.get(g) for g in primary]) if primary else 0.0
    # identity consistency over observations
    seg_obj = {}
    for r in rows:
        for s in r["segs"]:
            seg_obj[s.sid] = r["k"]
    prim_k = {g: r["k"] for g, r in primary.items()}
    obs = [s for s in all_segs if s.gt != 0]
    good = sum(1 for s in obs if seg_obj.get(s.sid) is not None and prim_k.get(s.gt) == seg_obj[s.sid])
    idc = good / max(len(obs), 1)
    switches = 0
    per_gt = defaultdict(list)
    for s in sorted(obs, key=lambda s: s.frame):
        if s.sid in seg_obj:
            per_gt[s.gt].append(seg_obj[s.sid])
    for seq in per_gt.values():
        switches += sum(1 for a, b in zip(seq, seq[1:]) if a != b)
    return dict(objects=len(rows), tp=tp, duplicates=dup, false_pos=fp, gt_assets=n_gt,
                precision=prec, recall=rec, f1=f1, duplicate_rate=dup / max(tp, 1),
                label_acc=float(lab_acc), identity_consistency=idc, id_switches=switches,
                mean_purity=float(np.mean([r["purity"] for r in primary.values()])) if primary else 0.0)


def _enc(P, res):
    ijk = np.floor(P / res).astype(np.int64) + (1 << 20)
    return (ijk[:, 0] << 42) | (ijk[:, 1] << 21) | ijk[:, 2]


def gt_voxel_labels(seq, frame_ids, instance_class, classes, res=0.1, stride=4, max_depth=8.0, include_ids=None):
    """Voxelise GT classes from GT poses (majority vote per voxel). Returns {voxel key: class} where class is
    an asset class, 'background', or 'ignore' (assets outside include_ids, e.g. not detectable)."""
    from .mapping import backproject
    names = list(classes) + ["background", "ignore"]
    cid = {c: k for k, c in enumerate(names)}
    lut_cache = {}
    K_all, C_all = [], []
    inc = set(include_ids) if include_ids is not None else None
    for i in frame_ids:
        f = seq.frame(i)
        P, z, (v, u) = backproject(f["depth"], seq.K, seq.gt_poses[i], stride=stride, max_depth=max_depth)
        ids = f["instance"][v, u]
        cl = np.full(len(ids), cid["background"], np.int64)
        for a in np.unique(ids):
            if a == 0:
                continue
            c = instance_class.get(int(a))
            if c is None:
                continue
            m = ids == a
            cl[m] = cid["ignore"] if (c not in cid or (inc is not None and int(a) not in inc)) else cid[c]
        K_all.append(_enc(P, res)); C_all.append(cl)
    keys = np.concatenate(K_all); cls = np.concatenate(C_all)
    up, cnt = np.unique(np.stack([keys, cls], 1), axis=0, return_counts=True)
    order = np.lexsort((-cnt, up[:, 0]))
    up = up[order]
    first = np.ones(len(up), bool); first[1:] = up[1:, 0] != up[:-1, 0]
    return {int(k): names[int(c)] for k, c in up[first]}


def semantic_miou(objects, gt_vox, classes, res=0.1, T_align=np.eye(4)):
    """3D IoU per class on voxels the GT observed; 'ignore' voxels are excluded."""
    pred = {}
    for label, segs, inst in objects:
        P = inst.pts if inst is not None else np.vstack([s.pts for s in segs])
        P = P @ T_align[:3, :3].T + T_align[:3, 3]
        for k in np.unique(_enc(P, res)):
            pred[int(k)] = label
    ious = {}
    for c in classes:
        g = {k for k, v in gt_vox.items() if v == c}
        p = {k for k, v in pred.items() if v == c and gt_vox.get(k, "ignore") != "ignore"}
        if not g and not p:
            continue
        ious[c] = len(g & p) / max(len(g | p), 1)
    return dict(per_class=ious, miou=float(np.mean(list(ious.values()))) if ious else 0.0)

In [ ]:
%%writefile /content/geosemtwinx/pipeline.py
"""End-to-end GeoSemTwin-X run: poses -> TSDF + occupancy -> GASF -> digital twin."""
import resource
import time

import numpy as np

from .gasf import GASF, make_segments
from .mapping import OccupancyMap, TSDFMap


def peak_ram_gb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024 ** 2


def run_mapping(seq, poses, sigmas, detector, classes, gasf_stride=5, tsdf_voxel=0.02, tsdf_depth=4.0,
                occ_bounds=None, occ_res=0.1, max_depth=8.0, gasf_kwargs=None, use_gt_eval=True,
                verbose=True, detector_on="frame", seg_kwargs=None, det_cache=None, map_sigma_max=0.03):
    """detector: callable (frame or rgb) -> detections, or None when det_cache {frame index: dets} is given."""
    f0 = seq.frame(0)
    h, w = f0["depth"].shape
    tsdf = TSDFMap(seq.K, w, h, voxel=tsdf_voxel, depth_max=tsdf_depth, sigma_max=map_sigma_max)
    if occ_bounds is None:
        C = np.array([T[:3, 3] for T in poses])
        occ_bounds = (C.min(0) - max_depth, C.max(0) + max_depth)
    occ = OccupancyMap(occ_bounds[0], occ_bounds[1], occ_res, max_range=max_depth)
    gasf = GASF(classes, **(gasf_kwargs or {}))
    segs_all, t = [], dict(tsdf=[], occ=[], detect=[], lift=[], fuse=[])
    gasf_frames = []
    for i in range(len(seq)):
        f = seq.frame(i)
        T, sg = poses[i], (sigmas[i] if sigmas is not None else 0.0)
        a = time.perf_counter(); tsdf.integrate(f, T, sg); t["tsdf"].append(time.perf_counter() - a)
        if i % gasf_stride:
            continue
        gasf_frames.append(i)
        a = time.perf_counter(); occ.integrate(f["depth"], seq.K, T, stride=8, pose_sigma=sg)
        t["occ"].append(time.perf_counter() - a)
        a = time.perf_counter()
        if det_cache is not None:
            dets = det_cache.get(i, [])
        else:
            dets = detector(f if detector_on == "frame" else f["rgb"])
        t["detect"].append(time.perf_counter() - a)
        a = time.perf_counter()
        segs = make_segments(f, dets, seq.K, T, classes, i, pose_sigma=sg, max_depth=max_depth,
                             gt_instance=f.get("instance") if use_gt_eval else None, start_sid=len(segs_all),
                             **(seg_kwargs or {}))
        t["lift"].append(time.perf_counter() - a)
        a = time.perf_counter(); gasf.update(segs, f, seq.K, T); t["fuse"].append(time.perf_counter() - a)
        segs_all += segs
        if verbose and len(gasf_frames) % 20 == 0:
            print(f"  frame {i}/{len(seq)}  segments={len(segs_all)}  instances={len(gasf.instances)}")
    timings = {k: float(np.mean(v)) * 1000 if v else 0.0 for k, v in t.items()}
    return dict(tsdf=tsdf, occ=occ, gasf=gasf, segments=segs_all, gasf_frames=gasf_frames,
                timings_ms=timings, peak_ram_gb=peak_ram_gb())


def visible_gt_assets(seq, frame_ids, min_pixels=None, min_frames=2, max_depth=8.0, min_frac=0.002):
    """GT assets a detector could reasonably find: in >= min_frames keyframes, each time covering
    >= min_pixels (default 0.2% of the image) with median depth <= max_depth."""
    cnt = {}
    for i in frame_ids:
        f = seq.frame(i)
        inst, depth = f["instance"], f["depth"]
        mp = min_pixels or int(min_frac * inst.size)
        ids, c = np.unique(inst, return_counts=True)
        for a, b in zip(ids, c):
            if not a or b < mp or a not in (seq.instance_class or {}):
                continue
            z = depth[inst == a]
            z = z[z > 0]
            if len(z) and np.median(z) <= max_depth:
                cnt[a] = cnt.get(a, 0) + 1
    return sorted(int(a) for a, n in cnt.items() if n >= min_frames)

In [ ]:
%%writefile /content/geosemtwinx/change.py
"""Structural / asset change detection between two digital-twin versions."""
import numpy as np
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree


def chamfer(A, B):
    if len(A) == 0 or len(B) == 0:
        return np.inf
    da, _ = cKDTree(B).query(A); db, _ = cKDTree(A).query(B)
    return float(0.5 * (np.median(da) + np.median(db)))


def detect_changes(assets_a, assets_b, occ_a=None, occ_b=None, move_thr=0.5, geom_thr=0.25, gate=4.0):
    """assets_*: list of dict(id, label, centroid, extent, pts).
    occ_*: OccupancyMap of each session. A change is only reported when the other session actually
    observed the relevant space: a move needs the old spot to be free in B (and the new spot free in A),
    a removal needs A's spot free in B, an addition needs B's spot free in A. Otherwise -> 'unverified'."""
    na, nb = len(assets_a), len(assets_b)
    C = np.full((na, nb), 1e3)
    for i, a in enumerate(assets_a):
        for j, b in enumerate(assets_b):
            if a["label"] != b["label"]:
                continue
            d = np.linalg.norm(a["centroid"] - b["centroid"])
            if d > gate:
                continue
            size = np.abs(np.log((a["extent"] + 0.05) / (b["extent"] + 0.05))).sum()
            C[i, j] = d + 0.5 * size
    rows, cols = linear_sum_assignment(C) if na and nb else ([], [])
    pairs = [(r, c) for r, c in zip(rows, cols) if C[r, c] < 1e3]
    out, ma, mb = [], set(), set()
    for r, c in pairs:
        a, b = assets_a[r], assets_b[c]
        d = float(np.linalg.norm(a["centroid"] - b["centroid"]))
        thr = max(move_thr, 0.4 * float(max(a["extent"].max(), b["extent"].max())))
        if d < thr:
            ma.add(r); mb.add(c)                                    # same asset, same place
            out.append(dict(type="unchanged", a=a["id"], b=b["id"], label=a["label"], displacement=d))
            ch = chamfer(a["pts"] - a["centroid"], b["pts"] - b["centroid"])
            ext = np.abs(a["extent"] - b["extent"]).max()
            if ch > geom_thr and ext > geom_thr:
                out.append(dict(type="geometry", a=a["id"], b=b["id"], label=a["label"], chamfer=ch))
        elif _observed_free(occ_b, a["pts"]) and _observed_free(occ_a, b["pts"]):
            ma.add(r); mb.add(c)                                    # verified move
            out.append(dict(type="moved", a=a["id"], b=b["id"], label=a["label"], displacement=d))
        # else: leave both unmatched; handled below as removed / added / unverified
    for i, a in enumerate(assets_a):
        if i in ma:
            continue
        if occ_b is None or _observed_free(occ_b, a["pts"]):
            out.append(dict(type="removed", a=a["id"], label=a["label"]))
        elif not _observed(occ_b, a["pts"]):
            out.append(dict(type="unverified", a=a["id"], label=a["label"]))
    for j, b in enumerate(assets_b):
        if j in mb:
            continue
        if occ_a is None or _observed_free(occ_a, b["pts"]):
            out.append(dict(type="added", b=b["id"], label=b["label"]))
        elif not _observed(occ_a, b["pts"]):
            out.append(dict(type="unverified", b=b["id"], label=b["label"]))
    return out


def _observed(occ, pts, min_frac=0.5):
    ijk, ok = occ.index(pts)
    if not ok.any():
        return False
    return (occ.state()[occ.flat(ijk[ok])] >= 0).mean() >= min_frac


def _observed_free(occ, pts, min_frac=0.5):
    """True if the other session observed this space and found it free (not occupied)."""
    if occ is None:
        return True
    ijk, ok = occ.index(pts)
    if not ok.any():
        return False
    st = occ.state()[occ.flat(ijk[ok])]
    observed = st >= 0
    return observed.mean() >= min_frac and (st[observed] == 0).mean() >= 0.5


def score_changes(pred, gt_changes, name_a, name_b):
    """pred items reference asset ids; name_* map asset id -> GT object name (or None)."""
    gt = {(c["name"], c["type"]) for c in gt_changes}
    P = set()
    for p in pred:
        if p["type"] in ("unchanged", "unverified"):
            continue
        if p["type"] in ("moved", "geometry"):
            n = name_a.get(p["a"]) or name_b.get(p["b"])
            P.add((n, "moved" if p["type"] == "moved" else "geometry"))
        elif p["type"] == "removed":
            P.add((name_a.get(p["a"]), "removed"))
        elif p["type"] == "added":
            P.add((name_b.get(p["b"]), "added"))
    tp = len(P & gt)
    prec = tp / max(len(P), 1); rec = tp / max(len(gt), 1)
    return dict(tp=tp, predicted=len(P), gt=len(gt), precision=prec, recall=rec,
                f1=2 * prec * rec / max(prec + rec, 1e-9), pred=sorted(P, key=str), gt_list=sorted(gt))

In [ ]:
%%writefile /content/geosemtwinx/twin.py
"""Persistent digital twin: versioned asset registry, spatial queries, store, 3D viewer."""
import datetime as _dt
import json
import os
from pathlib import Path

import numpy as np


def _r(x, n=3):
    return [round(float(v), n) for v in np.asarray(x).ravel()]


class DigitalTwin:
    def __init__(self, name="GeoSemTwin-X"):
        self.name = name
        self.version = 0
        self.assets = {}          # id -> record
        self.history = []         # list of version summaries
        self.changes = []
        self._pts = {}            # asset id -> world points (kept in memory for change detection)
        self._occ = None

    # ------------------------------------------------------------ build / update
    def commit(self, instances, classes, session, T_align=np.eye(4), changes=None, map_stats=None):
        """instances: confirmed GASF Instance objects (estimated world frame)."""
        self.version += 1
        ts = _dt.datetime.now(_dt.timezone.utc).isoformat(timespec="seconds")
        seen = set()
        for inst in instances:
            P = inst.pts @ T_align[:3, :3].T + T_align[:3, 3]
            aid = getattr(inst, "asset_id", None) or f"A{len(self.assets) + 1:04d}"
            inst.asset_id = aid
            lo, hi = np.percentile(P, [2, 98], axis=0)           # robust box: ignore stray points
            core = P[np.all((P >= lo) & (P <= hi), axis=1)]
            P = core if len(core) >= 10 else P
            rec = self.assets.get(aid, dict(id=aid, first_seen=ts, versions=[]))
            rec.update(label=classes[int(np.argmax(inst.probs))], confidence=round(float(inst.probs.max()), 3),
                       label_entropy=round(inst.label_entropy(), 3), centroid=_r(np.median(P, 0)),
                       bbox_min=_r(lo), bbox_max=_r(hi), size=_r(hi - lo), observations=int(inst.hits),
                       views=len(inst.views), last_seen=ts, status="present", session=session)
            rec["versions"].append(dict(version=self.version, centroid=rec["centroid"], size=rec["size"]))
            self.assets[aid] = rec
            self._pts[aid] = P
            seen.add(aid)
        for c in changes or []:
            c = dict(c, version=self.version, time=ts)
            self.changes.append(c)
            if c["type"] == "removed" and c.get("a") in self.assets:
                self.assets[c["a"]]["status"] = "removed"
        self.history.append(dict(version=self.version, time=ts, session=session, n_assets=len(seen),
                                 map=map_stats or {}))
        return self.version

    def update(self, instances, classes, session, occ_prev=None, occ_new=None, T_align=np.eye(4)):
        """Commit a new mapping session against the current twin: assets keep their IDs when re-observed
        (moved ones are re-positioned), new ones get new IDs, removals are verified with occupancy."""
        from .change import detect_changes
        if self.version == 0:
            v = self.commit(instances, classes, session, T_align)
            self._occ = occ_new
            return v, []
        prev = [dict(id=a["id"], label=a["label"], centroid=np.array(a["centroid"]), extent=np.array(a["size"]),
                     pts=self._pts[a["id"]]) for a in self.present() if a["id"] in self._pts]
        new = []
        for k, inst in enumerate(instances):
            P = inst.pts @ T_align[:3, :3].T + T_align[:3, 3]
            new.append(dict(id=f"new{k}", label=classes[int(np.argmax(inst.probs))], centroid=P.mean(0),
                            extent=np.ptp(P, 0), pts=P, inst=inst))
        ch = detect_changes(prev, new, occ_prev if occ_prev is not None else getattr(self, "_occ", None), occ_new)
        keep = {}
        for c in ch:
            if c["type"] in ("unchanged", "moved", "geometry"):
                keep[c["b"]] = c["a"]
        for n in new:
            n["inst"].asset_id = keep.get(n["id"])            # None -> commit() assigns a fresh ID
        log = []
        for c in ch:
            if c["type"] == "unchanged":
                continue
            e = dict(c)
            if "b" in e:
                e["b"] = keep.get(e["b"], e["b"])
            log.append(e)
        v = self.commit(instances, classes, session, T_align, changes=[])
        id_of = {n["id"]: n["inst"].asset_id for n in new}
        for e in log:
            if e["type"] in ("added", "unverified") and "b" in e:
                e["b"] = id_of.get(e["b"], e["b"])
            e.update(version=self.version, time=self.history[-1]["time"])
            self.changes.append(e)
            if e["type"] == "removed" and e.get("a") in self.assets:
                self.assets[e["a"]]["status"] = "removed"
        self._occ = occ_new
        return v, log

    # ------------------------------------------------------------ spatial queries
    def present(self):
        return [a for a in self.assets.values() if a["status"] == "present"]

    def query_radius(self, point, radius, label=None):
        p = np.asarray(point, float)
        out = []
        for a in self.present():
            if label and a["label"] != label:
                continue
            d = float(np.linalg.norm(np.array(a["centroid"]) - p))
            if d <= radius:
                out.append((d, a))
        return [a for _, a in sorted(out, key=lambda x: x[0])]

    def query_box(self, lo, hi, label=None):
        lo, hi = np.asarray(lo), np.asarray(hi)
        return [a for a in self.present() if (label is None or a["label"] == label)
                and np.all(np.array(a["centroid"]) >= lo) and np.all(np.array(a["centroid"]) <= hi)]

    def nearest(self, point, label):
        c = [a for a in self.present() if a["label"] == label]
        if not c:
            return None
        return min(c, key=lambda a: np.linalg.norm(np.array(a["centroid"]) - np.asarray(point)))

    def counts(self):
        out = {}
        for a in self.present():
            out[a["label"]] = out.get(a["label"], 0) + 1
        return out

    # ------------------------------------------------------------ persistence
    def to_dict(self):
        return dict(name=self.name, version=self.version, assets=list(self.assets.values()),
                    history=self.history, changes=self.changes)


class TwinStore:
    """Versioned store. Point `root` at a synced folder (e.g. a mounted Google Drive
    path in Colab, or an S3/GCS fuse mount) to get cloud synchronisation."""

    def __init__(self, root):
        self.root = Path(root); self.root.mkdir(parents=True, exist_ok=True)

    def push(self, twin):
        d = twin.to_dict()
        p = self.root / f"twin_v{twin.version:03d}.json"
        p.write_text(json.dumps(d, indent=1))
        idx = self.root / "index.json"
        index = json.loads(idx.read_text()) if idx.exists() else []
        index.append(dict(version=twin.version, file=p.name, n_assets=len(twin.present()),
                          time=twin.history[-1]["time"]))
        idx.write_text(json.dumps(index, indent=1))
        return p

    def versions(self):
        idx = self.root / "index.json"
        return json.loads(idx.read_text()) if idx.exists() else []

    def load(self, version):
        return json.loads((self.root / f"twin_v{version:03d}.json").read_text())


# ---------------------------------------------------------------- viewer
CLASS_COLORS = {"pallet": "#b07a3c", "cardboard box": "#d9a35b", "forklift": "#f2b705", "barrel": "#2e8b57",
                "rack": "#3366cc", "monitor": "#555", "keyboard": "#888", "cup": "#c33", "chair": "#836",
                "book": "#365", "mouse": "#999", "laptop": "#444"}


def _box_lines(lo, hi):
    x0, y0, z0 = lo; x1, y1, z1 = hi
    c = [(x0, y0, z0), (x1, y0, z0), (x1, y1, z0), (x0, y1, z0), (x0, y0, z1), (x1, y0, z1), (x1, y1, z1), (x0, y1, z1)]
    e = [(0, 1), (1, 2), (2, 3), (3, 0), (4, 5), (5, 6), (6, 7), (7, 4), (0, 4), (1, 5), (2, 6), (3, 7)]
    X, Y, Z = [], [], []
    for a, b in e:
        X += [c[a][0], c[b][0], None]; Y += [c[a][1], c[b][1], None]; Z += [c[a][2], c[b][2], None]
    return X, Y, Z


def viewer_html(path, twin, map_points=None, map_colors=None, traj_est=None, traj_gt=None, title=None,
                max_points=60000):
    import plotly.graph_objects as go
    fig = go.Figure()
    if map_points is not None and len(map_points):
        idx = np.random.default_rng(0).choice(len(map_points), min(max_points, len(map_points)), replace=False)
        P = map_points[idx]
        col = None
        if map_colors is not None:
            Cc = (np.asarray(map_colors)[idx] * 255).astype(int)
            col = [f"rgb({r},{g},{b})" for r, g, b in Cc]
        fig.add_trace(go.Scatter3d(x=P[:, 0], y=P[:, 1], z=P[:, 2], mode="markers", name="map",
                                   marker=dict(size=1.2, color=col or "#9aa", opacity=0.6), hoverinfo="skip"))
    for a in twin.present():
        X, Y, Z = _box_lines(a["bbox_min"], a["bbox_max"])
        col = CLASS_COLORS.get(a["label"], "#e4572e")
        fig.add_trace(go.Scatter3d(x=X, y=Y, z=Z, mode="lines", line=dict(color=col, width=5),
                                   name=f'{a["id"]} {a["label"]}', legendgroup=a["label"], showlegend=False,
                                   hoverinfo="skip"))
        c = a["centroid"]
        fig.add_trace(go.Scatter3d(
            x=[c[0]], y=[c[1]], z=[a["bbox_max"][2] + 0.15], mode="markers+text", text=[a["id"]],
            textposition="top center", marker=dict(size=4, color=col), name=a["label"], legendgroup=a["label"],
            showlegend=False,
            hovertext=[f'{a["id"]} | {a["label"]} ({a["confidence"]:.2f})<br>size {a["size"]} m<br>'
                       f'obs {a["observations"]}, views {a["views"]}'], hoverinfo="text"))
    for traj, name, color in ((traj_gt, "ground-truth path", "#2a9d8f"), (traj_est, "estimated path", "#e76f51")):
        if traj is not None and len(traj):
            T = np.asarray(traj)
            fig.add_trace(go.Scatter3d(x=T[:, 0], y=T[:, 1], z=T[:, 2], mode="lines", name=name,
                                       line=dict(color=color, width=4)))
    for ch in twin.changes:
        if ch["type"] in ("moved", "added", "removed") and ("b" in ch or "a" in ch):
            aid = ch.get("b") or ch.get("a")
            if aid in twin.assets:
                c = twin.assets[aid]["centroid"]
                sym = {"moved": "diamond", "added": "circle", "removed": "x"}[ch["type"]]
                fig.add_trace(go.Scatter3d(x=[c[0]], y=[c[1]], z=[c[2] + 1.2], mode="markers+text",
                                           text=[ch["type"]], marker=dict(size=7, symbol=sym, color="#d62828"),
                                           showlegend=False))
    fig.update_layout(title=title or f"{twin.name} v{twin.version}", scene=dict(aspectmode="data"),
                      margin=dict(l=0, r=0, t=40, b=0), legend=dict(itemsizing="constant"))
    fig.write_html(path, include_plotlyjs="cdn")
    return path

In [ ]:
%%writefile /content/geosemtwinx/nbv.py
"""Uncertainty-guided next-best-view exploration with LiDAR obstacle checks and
local replanning, plus frontier / random baselines (run in the synthetic simulator)."""
import heapq

import numpy as np
from scipy import ndimage

from .mapping import OccupancyMap


def dijkstra(free, start):
    """8-connected grid distances (cells) from start over free cells."""
    H, W = free.shape
    dist = np.full((H, W), np.inf); prev = -np.ones((H, W, 2), int)
    if not free[start]:
        return dist, prev
    dist[start] = 0; pq = [(0.0, start)]
    nb = [(-1, 0, 1), (1, 0, 1), (0, -1, 1), (0, 1, 1), (-1, -1, 1.414), (-1, 1, 1.414), (1, -1, 1.414), (1, 1, 1.414)]
    while pq:
        d, (i, j) = heapq.heappop(pq)
        if d > dist[i, j]:
            continue
        for di, dj, c in nb:
            a, b = i + di, j + dj
            if 0 <= a < H and 0 <= b < W and free[a, b] and d + c < dist[a, b]:
                dist[a, b] = d + c; prev[a, b] = (i, j); heapq.heappush(pq, (d + c, (a, b)))
    return dist, prev


def backtrack(prev, goal):
    path = [goal]
    while True:
        p = tuple(prev[path[-1]])
        if p[0] < 0:
            break
        path.append(p)
    return path[::-1]


def camera_pose(x, y, yaw, height=1.4, pitch_deg=-12):
    from .synthetic import look_at
    d = np.array([np.cos(yaw), np.sin(yaw), np.tan(np.radians(pitch_deg))])
    eye = np.array([x, y, height])
    return look_at(eye, eye + d)


class Explorer:
    def __init__(self, sim, strategy="nbv", res=0.1, bounds=((-0.5, -0.5, -0.2), (24.5, 16.5, 6.0)),
                 robot_radius=0.35, seed=0, gasf=None, detector=None, classes=None):
        self.sim, self.strategy = sim, strategy
        self.occ = OccupancyMap(bounds[0], bounds[1], res, max_range=7.0)
        self.r_cells = int(np.ceil(robot_radius / res))
        self.rng = np.random.default_rng(seed)
        self.gasf, self.det, self.classes = gasf, detector, classes
        self.asset_pixels = {}
        self.log = []
        self.path_len = 0.0
        self.replans = 0
        self.collisions = 0
        self.seg_counter = 0
        self.visited_goals = []
        # GT surface for coverage
        import open3d as o3d
        pts = []
        for o in sim.objs:
            if o["cls"] in ("floor",):
                continue
            pc = o["mesh"].sample_points_uniformly(int(max(200, o["mesh"].get_surface_area() * 40)))
            pts.append(np.asarray(pc.points))
        P = np.vstack(pts)
        P = P[P[:, 2] < 4.5]
        ijk, ok = self.occ.index(P)
        self.gt_surface = np.unique(self.occ.flat(ijk[ok]))
        # GT 2D obstacle map for collision accounting
        occ_gt = np.zeros(self.occ.shape[:2], bool)
        Q = P[(P[:, 2] > 0.1) & (P[:, 2] < 1.8)]
        ij, ok2 = self.occ.index(Q)
        occ_gt[ij[ok2, 0], ij[ok2, 1]] = True
        self.gt_obst = occ_gt

    # ---------------------------------------------------------------- sensing
    def lidar(self, x, y, z=0.35, n=360, rmax=8.0):
        import open3d as o3d
        ang = np.linspace(0, 2 * np.pi, n, endpoint=False)
        d = np.stack([np.cos(ang), np.sin(ang), np.zeros(n)], 1)
        rays = np.concatenate([np.tile([x, y, z], (n, 1)), d], 1).astype(np.float32)
        t = self.sim.scene.cast_rays(o3d.core.Tensor(rays))["t_hit"].numpy()
        hit = np.isfinite(t) & (t < rmax)
        return np.array([x, y, z]) + d[hit] * t[hit, None]

    def sense(self, x, y, yaw, frame_idx):
        T = camera_pose(x, y, yaw)
        f = self.sim.render(T)
        self.occ.integrate(f["depth"], self.sim.K, T, stride=6)
        for z in (0.08, 0.3, 0.6, 1.0):                             # multi-ring LiDAR
            L = self.lidar(x, y, z=z)
            self.occ.integrate_points(L, np.array([x, y, z]))
        ids, cnt = np.unique(f["instance"], return_counts=True)
        for i, c in zip(ids, cnt):
            if i:
                self.asset_pixels[i] = self.asset_pixels.get(i, 0) + c
        if self.gasf is not None:
            from .gasf import make_segments
            segs = make_segments(f, self.det(f), self.sim.K, T, self.classes, frame_idx,
                                 gt_instance=f["instance"], start_sid=self.seg_counter)
            self.seg_counter += len(segs)
            self.gasf.update(segs, f, self.sim.K, T)

    # ---------------------------------------------------------------- planning
    def nav_grid(self):
        trav = self.occ.traversability(floor_z=0.0, robot_height=1.8)
        blocked = ndimage.binary_dilation(trav == 0, iterations=self.r_cells)
        return (trav == 1) & ~blocked

    def ambiguous_voxels(self):
        if self.gasf is None:
            return np.empty(0, np.int64)
        P = [i.pts for i in self.gasf.instances.values()
             if (not self.gasf.is_confirmed(i)) or i.label_entropy() > 0.4]
        if not P:
            return np.empty(0, np.int64)
        ijk, ok = self.occ.index(np.vstack(P))
        return np.unique(self.occ.flat(ijk[ok]))

    def view_gain(self, x, y, yaw, state, sigma, amb, hfov=62, vfov=48, nh=20, nv=10, rng_max=6.0):
        T = camera_pose(x, y, yaw)
        a = np.radians(np.linspace(-hfov / 2, hfov / 2, nh)); b = np.radians(np.linspace(-vfov / 2, vfov / 2, nv))
        A, B = np.meshgrid(a, b)
        dirs_c = np.stack([np.tan(A.ravel()), np.tan(B.ravel()), np.ones(A.size)], 1)
        dirs_c /= np.linalg.norm(dirs_c, axis=1, keepdims=True)
        dirs = dirs_c @ T[:3, :3].T
        steps = np.arange(0.3, rng_max, self.occ.res)
        S = T[:3, 3] + dirs[:, None, :] * steps[None, :, None]
        ijk, ok = self.occ.index(S.reshape(-1, 3))
        flat = np.where(ok, self.occ.flat(np.where(ok[:, None], ijk, 0)), -1).reshape(len(dirs), len(steps))
        st = np.where(flat >= 0, state[np.maximum(flat, 0)], 1)
        occ_hit = st == 1
        first = np.where(occ_hit.any(1), occ_hit.argmax(1), len(steps))
        idx = np.arange(len(steps))[None, :]
        before = idx < first[:, None]
        unknown = np.unique(flat[before & (st == -1)])
        hit_flat = flat[np.arange(len(dirs)), np.minimum(first, len(steps) - 1)][first < len(steps)]
        unc = np.sum(sigma[hit_flat] > 0.02) if len(hit_flat) else 0
        sem = np.isin(hit_flat, amb).sum() if len(amb) else 0
        return len(unknown), unc, sem

    def choose_goal(self, ri, rj, nav):
        dist, prev = dijkstra(nav, (ri, rj))
        reach = np.argwhere(np.isfinite(dist) & (dist > 3))
        if len(reach) == 0:
            return None, None, dist, prev
        state = self.occ.state(); sigma = self.occ.surface_sigma()
        if self.strategy == "random":
            g = tuple(reach[self.rng.integers(len(reach))])
            return g, self.rng.uniform(0, 2 * np.pi), dist, prev
        st2 = state.reshape(self.occ.shape)
        unknown_col = (st2[:, :, 1:20] == -1).sum(axis=2) > 5
        if self.strategy == "frontier":
            front = nav & ndimage.binary_dilation(unknown_col, iterations=2) & np.isfinite(dist)
            cand = np.argwhere(front & (dist > 3))
            if len(cand) and self.visited_goals:                    # classic frontier blacklisting
                V = np.array(self.visited_goals)
                dmin = np.min(np.linalg.norm(cand[:, None, :] - V[None, :, :], axis=2), axis=1)
                cand = cand[dmin * self.occ.res > 1.5]
            if len(cand) == 0:
                g = tuple(reach[self.rng.integers(len(reach))]); return g, self.rng.uniform(0, 2 * np.pi), dist, prev
            g = tuple(cand[np.argmin(dist[tuple(cand.T)])])
            uc = np.argwhere(unknown_col)
            near = uc[np.argmin(np.linalg.norm(uc - np.array(g), axis=1))] if len(uc) else np.array(g) + 1
            return g, float(np.arctan2(near[1] - g[1], near[0] - g[0])), dist, prev
        # NBV: sample candidates, evaluate information gain of 8 headings
        amb = self.ambiguous_voxels()
        cand = reach[self.rng.choice(len(reach), min(60, len(reach)), replace=False)]
        best, bu = None, -1
        for (i, j) in cand:
            x, y = self.occ.center(np.array([[i, j, 0]]))[0][:2]
            for yaw in np.linspace(0, 2 * np.pi, 8, endpoint=False):
                gu, gs, gm = self.view_gain(x, y, yaw, state, sigma, amb)
                u = (gu + 2.0 * gs + 4.0 * gm) * np.exp(-0.15 * dist[i, j] * self.occ.res)
                if u > bu:
                    best, bu = ((i, j), yaw), u
        return best[0], best[1], dist, prev

    # ---------------------------------------------------------------- main loop
    def run(self, start=(3.5, 8.0, 0.0), budget_m=80.0, max_steps=400, step_m=0.5):
        x, y, yaw = start
        k = 0
        for a in np.linspace(0, 2 * np.pi, 4, endpoint=False):       # initial look-around
            self.sense(x, y, yaw + a, k); k += 1
        self._log(k)
        while self.path_len < budget_m and k < max_steps:
            nav = self.nav_grid()
            ri, rj, _ = self.occ.index(np.array([[x, y, 0.5]]))[0][0]
            nav[max(ri - 1, 0):ri + 2, max(rj - 1, 0):rj + 2] = True
            goal, gyaw, dist, prev = self.choose_goal(ri, rj, nav)
            if goal is None:
                break
            self.visited_goals.append(goal)
            path = backtrack(prev, goal)
            pos = np.array([x, y]); acc = 0.0
            for n, cell in enumerate(path[1:], 1):
                c = self.occ.center(np.array([[cell[0], cell[1], 0]]))[0][:2]
                seg = np.linalg.norm(c - pos); acc += seg; self.path_len += seg
                heading = np.arctan2(c[1] - pos[1], c[0] - pos[0])
                pos = c
                if self.gt_obst[cell[0], cell[1]]:
                    self.collisions += 1
                if acc >= step_m or n == len(path) - 1:
                    acc = 0.0
                    look = gyaw if n == len(path) - 1 else heading
                    self.sense(pos[0], pos[1], look, k); k += 1
                    self._log(k)
                    # local replanning: LiDAR/camera found an obstacle on the remaining path
                    nav2 = self.nav_grid()
                    rest = path[n + 1:]
                    if any(not nav2[c2] for c2 in rest):
                        self.replans += 1
                        break
                if self.path_len >= budget_m:
                    break
            x, y = pos
            yaw = gyaw
        return self.log

    def _log(self, k):
        seen = sum(1 for v in self.asset_pixels.values() if v > 1500)
        self.log.append(dict(step=k, path_m=self.path_len, coverage=self.occ.coverage(self.gt_surface),
                             assets_seen=seen, replans=self.replans, collisions=self.collisions))

In [ ]:
%%writefile /content/geosemtwinx/bench.py
"""Synthetic-warehouse benchmark (CPU, exact ground truth): SLAM, GASF vs baselines, change detection, NBV."""
import copy
import json
import sys
import time
from pathlib import Path

import numpy as np

from .change import detect_changes, score_changes
from .data import SyntheticSequence
from .detectors import OracleNoisyDetector
from .evaluation import evaluate_objects, gt_voxel_labels, object_gt, semantic_miou
from .gasf import (GASF, baseline_centroid, baseline_geometry_clustering, baseline_independent)
from .metrics import ate, path_length, rpe
from .pipeline import run_mapping, visible_gt_assets
from .slam import run_slam
from .synthetic import ASSET_CLASSES

OUT = Path("results")


def set_out(path):
    global OUT
    OUT = Path(path); OUT.mkdir(parents=True, exist_ok=True)
BOUNDS = ((-0.5, -0.5, -0.2), (24.5, 16.5, 6.0))


def save(name, obj):
    OUT.mkdir(parents=True, exist_ok=True)
    def conv(o):
        if isinstance(o, (np.floating, np.integer)):
            return o.item()
        if isinstance(o, np.ndarray):
            return o.tolist()
        raise TypeError(type(o))
    (OUT / f"{name}.json").write_text(json.dumps(obj, indent=1, default=conv))


def exp_slam(n=720, scale=1.0):
    seq = SyntheticSequence("A", n, stride=2, scale=scale)
    n = len(seq)
    T0 = np.linalg.inv(seq.gt_poses[0]); gt = [T0 @ T for T in seq.gt_poses]
    t = time.time()
    slam, _ = run_slam(seq, depth_max=8.0)
    el = time.time() - t
    res = dict(frames=n, resolution=list(seq.frame(0)["depth"].shape), path_m=path_length(gt),
               keyframes=len(slam.keyframes), loops=slam.n_loops, track_fallbacks=slam.n_track_fail,
               time_per_frame_ms=1000 * el / n, opt_time_s=slam.t_opt)
    a = ate(slam.poses, gt); r = rpe(slam.poses, gt, seq.timestamps)
    a0 = ate(slam.poses_before_opt, gt); r0 = rpe(slam.poses_before_opt, gt, seq.timestamps)
    res.update(ate_rmse=a["rmse"], ate_median=a["median"], ate_max=a["max"], rpe_trans=r["trans_rmse"],
               rpe_rot_deg=r["rot_rmse_deg"], odo_ate_rmse=a0["rmse"], odo_rpe_trans=r0["trans_rmse"],
               drift_pct=100 * a["rmse"] / path_length(gt))
    save("slam", res)
    np.savez(OUT / "slam_traj.npz", est=np.array([T for T in slam.poses]), odo=np.array(slam.poses_before_opt),
             gt=np.array(gt), sig=np.array(slam.sigmas), sig_step=np.array(slam.step_sigmas), T0=np.linalg.inv(T0))
    print(json.dumps(res, indent=1))


def _semantic_run(seq, poses, sigmas, seed, gasf_kwargs=None, stride=3):
    det = OracleNoisyDetector(ASSET_CLASSES, seq.instance_class, seed=seed)
    return run_mapping(seq, poses, sigmas, det, ASSET_CLASSES, gasf_stride=stride, occ_bounds=BOUNDS,
                       gasf_kwargs=gasf_kwargs, verbose=False)


def exp_semantic(n=720, scale=0.5, seeds=(0, 1, 2), use_slam=True):
    seq = SyntheticSequence("A", n, stride=2, scale=scale)
    n = len(seq)
    poses, sigmas, T_align = seq.gt_poses, None, np.eye(4)
    if use_slam and (OUT / "slam_traj.npz").exists():
        d = np.load(OUT / "slam_traj.npz")
        T0inv = d["T0"]
        poses = [T0inv @ T for T in d["est"]][:n]            # SLAM poses in GT world frame (anchored at frame 0)
        sigmas = d["sig_step"][:n] if "sig_step" in d else None
    frames = list(range(0, n, 3))
    gt_vox = gt_voxel_labels(seq, frames, seq.instance_class, ASSET_CLASSES)
    methods = {"GASF (full)": {}, "GASF w/o geometry terms": dict(use_geometry=False),
               "GASF w/o multi-view gate": dict(use_multiview=False), "GASF w/o duplicate merge": dict(merge=False)}
    rows = []
    for seed in seeds:
        out = None
        for name, kw in methods.items():
            o = _semantic_run(seq, poses, sigmas, seed, kw)
            out = out or o
            vis = visible_gt_assets(seq, o["gasf_frames"])
            m = evaluate_objects(o["gasf"].objects(), o["segments"], seq.instance_class, vis)
            m.update(semantic_miou(o["gasf"].objects(), gt_vox, ASSET_CLASSES))
            m.update(method=name, seed=seed, timings=o["timings_ms"])
            rows.append(m); print(name, seed, round(m["f1"], 3), round(m["identity_consistency"], 3), round(m["miou"], 3))
        segs = out["segments"]; vis = visible_gt_assets(seq, out["gasf_frames"])
        for name, fn in (("Independent 2D labelling", baseline_independent), ("Centroid tracking", baseline_centroid),
                         ("Geometry-only clustering", baseline_geometry_clustering)):
            objs = fn(segs, ASSET_CLASSES)
            m = evaluate_objects(objs, segs, seq.instance_class, vis)
            m.update(semantic_miou(objs, gt_vox, ASSET_CLASSES))
            m.update(method=name, seed=seed); rows.append(m)
            print(name, seed, round(m["f1"], 3), round(m["identity_consistency"], 3), round(m["miou"], 3))
    save("semantic_slam" if use_slam else "semantic_gtpose", rows)


def _assets(gasf, seq):
    out, names = [], {}
    for k, (label, segs, inst) in enumerate(gasf.objects()):
        g, _ = object_gt(segs)
        aid = f"A{k:03d}"
        out.append(dict(id=aid, label=label, centroid=inst.pts.mean(0), extent=np.ptp(inst.pts, 0), pts=inst.pts))
        names[aid] = seq.instance_names[g] if g else None
    return out, names


def exp_change(n=720, scale=0.5, seeds=(0, 1, 2)):
    rows = []
    seqA = SyntheticSequence("A", n, stride=2, scale=scale); seqB = SyntheticSequence("B", n, stride=2, scale=scale)
    for seed in seeds:
        oA = _semantic_run(seqA, seqA.gt_poses, None, seed)
        oB = _semantic_run(seqB, seqB.gt_poses, None, seed + 100)
        aA, nA = _assets(oA["gasf"], seqA); aB, nB = _assets(oB["gasf"], seqB)
        pred = detect_changes(aA, aB, oA["occ"], oB["occ"])
        sc = score_changes(pred, seqB.changes, nA, nB)
        sc_no_vis = score_changes(detect_changes(aA, aB, None, None), seqB.changes, nA, nB)
        rows.append(dict(seed=seed, **{k: v for k, v in sc.items()},
                         no_visibility=dict(precision=sc_no_vis["precision"], recall=sc_no_vis["recall"],
                                            f1=sc_no_vis["f1"]),
                         raw=[{k: v for k, v in p.items()} for p in pred]))
        print(seed, sc["precision"], sc["recall"], sc["pred"], "| no-vis", sc_no_vis["precision"], sc_no_vis["recall"])
    save("change", rows)


def exp_nbv(budget=80.0, seeds=(0, 1, 2), strategies=("nbv", "frontier", "random")):
    from .nbv import Explorer
    from .synthetic import SyntheticWarehouse, default_layout
    rows = {}
    starts = [(3.5, 8.0, 0.0), (20.0, 8.0, np.pi), (12.0, 2.2, 0.0)]
    for seed in seeds:
        for st in strategies:
            objs = default_layout(0)
            sim = SyntheticWarehouse(objs, width=320, height=240, fx=262.5, fy=262.5, cx=159.5, cy=119.5, seed=seed)
            ic = {sim.inst_id[o["name"]]: o["cls"] for o in objs if o["cls"] in ASSET_CLASSES}
            g = GASF(ASSET_CLASSES); det = OracleNoisyDetector(ASSET_CLASSES, ic, seed=seed)
            ex = Explorer(sim, st, seed=seed, gasf=g, detector=det, classes=ASSET_CLASSES)
            t = time.time(); log = ex.run(start=starts[seed % 3], budget_m=budget)
            confirmed = len([i for i in g.instances.values() if g.is_confirmed(i)])
            rows[f"{st}_{seed}"] = dict(strategy=st, seed=seed, log=log, total_assets=len(ic),
                                        confirmed_instances=confirmed, time_s=time.time() - t)
            print(st, seed, "cov", round(log[-1]["coverage"], 3), "assets", log[-1]["assets_seen"], "/", len(ic),
                  "replans", log[-1]["replans"], "coll", log[-1]["collisions"], round(time.time() - t, 1), "s")
            save("nbv", rows)

In [ ]:
%%writefile /content/geosemtwinx/compare.py
"""Comparative studies.

    cache_detections   run the (expensive) 2D detector once per keyframe and reuse it everywhere
    fusion_study       GASF vs conventional association methods + GASF ablations
    pose_noise_study   robustness of every method to pose error (what SLAM drift does to the twin)
    threshold_study    sensitivity to the detector confidence threshold
    slam_study         SLAM configurations / ablations on any sequences, next to published numbers
    plots + tables     PNG charts and markdown tables for the report / dashboard
"""
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from .evaluation import evaluate_objects, gt_voxel_labels, semantic_miou
from .gasf import GASF, baseline_centroid, baseline_geometry_clustering, baseline_independent, make_segments
from .metrics import ate, path_length, rpe
from .pipeline import visible_gt_assets
from .slam import run_slam

# ATE RMSE (m) as reported in ORB-SLAM2 (Mur-Artal & Tardos, T-RO 2017, Table III) for the RGB-D methods.
PUBLISHED_ATE = {
    "rgbd_dataset_freiburg1_desk": {"ORB-SLAM2": 0.016, "ElasticFusion": 0.020, "DVO-SLAM": 0.021},
    "rgbd_dataset_freiburg2_xyz": {"ORB-SLAM2": 0.004, "ElasticFusion": 0.011, "DVO-SLAM": 0.018},
    "rgbd_dataset_freiburg3_long_office_household": {"ORB-SLAM2": 0.010, "ElasticFusion": 0.017, "DVO-SLAM": 0.035},
}

METHOD_ORDER = ["GASF (ours)", "GASF w/o geometry terms", "GASF w/o multi-view gate", "GASF w/o duplicate merge",
                "Centroid tracking", "Geometry-only clustering", "Independent 2D labelling"]
PALETTE = {"GASF (ours)": "#1F5BD8", "Centroid tracking": "#D9731A", "Geometry-only clustering": "#2A9D7F",
           "Independent 2D labelling": "#A04FB0", "GASF w/o geometry terms": "#7FA3EA",
           "GASF w/o multi-view gate": "#9DB8EE", "GASF w/o duplicate merge": "#BACDF3"}
GASF_VARIANTS = {"GASF (ours)": {}, "GASF w/o geometry terms": dict(use_geometry=False),
                 "GASF w/o multi-view gate": dict(use_multiview=False), "GASF w/o duplicate merge": dict(merge=False)}
BASELINES = {"Centroid tracking": baseline_centroid, "Geometry-only clustering": baseline_geometry_clustering,
             "Independent 2D labelling": baseline_independent}
KEYS = ["objects", "precision", "recall", "f1", "duplicate_rate", "identity_consistency", "id_switches",
        "label_acc", "mean_purity", "miou"]


# ---------------------------------------------------------------- detections
def cache_detections(seq, detector, frames, on="rgb", verbose=True):
    cache, t = {}, []
    for n, i in enumerate(frames):
        f = seq.frame(i)
        a = time.perf_counter()
        cache[i] = detector(f["rgb"] if on == "rgb" else f)
        t.append(time.perf_counter() - a)
        if verbose and n % 25 == 0:
            print(f"  detections {n}/{len(frames)}")
    cache_ms = 1000 * float(np.mean(t)) if t else 0.0
    return cache, cache_ms


def save_cache(path, cache):
    out = {}
    for i, dets in cache.items():
        for k, d in enumerate(dets):
            out[f"{i}_{k}_mask"] = np.packbits(d["mask"])
            out[f"{i}_{k}_meta"] = np.array([d["score"], d.get("mask_score", 1.0)] + list(d["mask"].shape), float)
            out[f"{i}_{k}_label"] = np.array(d["label"])
    np.savez_compressed(path, **out)


def load_cache(path):
    z = np.load(path, allow_pickle=False)
    cache = {}
    metas = sorted((tuple(map(int, key.split("_")[:2])), key) for key in z.files if key.endswith("_meta"))
    for (i, k), key in metas:
        meta = z[key]; h, w = int(meta[2]), int(meta[3])
        mask = np.unpackbits(z[f"{i}_{k}_mask"])[: h * w].reshape(h, w).astype(bool)
        cache.setdefault(i, []).append(dict(label=str(z[f"{i}_{k}_label"]), score=float(meta[0]),
                                            mask_score=float(meta[1]), mask=mask))
    return cache


# ---------------------------------------------------------------- fusion
def lift(seq, cache, frames, poses, classes, sigmas=None, seg_kwargs=None, max_depth=8.0, use_gt=True):
    by_frame, n = {}, 0
    for i in frames:
        f = seq.frame(i)
        segs = make_segments(f, cache.get(i, []), seq.K, poses[i], classes, i,
                             pose_sigma=sigmas[i] if sigmas is not None else 0.0, max_depth=max_depth,
                             gt_instance=f.get("instance") if use_gt else None, start_sid=n, **(seg_kwargs or {}))
        by_frame[i] = segs; n += len(segs)
    return by_frame


def fuse(seq, by_frame, frames, poses, classes, **kw):
    g = GASF(classes, **kw)
    for i in frames:
        g.update(by_frame[i], seq.frame(i), seq.K, poses[i])
    return g


def evaluate_all(seq, by_frame, frames, poses, classes, visible, gt_vox=None, methods=None):
    segs = [s for i in frames for s in by_frame[i]]
    rows = {}
    for name in (methods or METHOD_ORDER):
        if name in GASF_VARIANTS:
            objs = fuse(seq, by_frame, frames, poses, classes, **GASF_VARIANTS[name]).objects()
        else:
            objs = BASELINES[name](segs, classes)
        m = evaluate_objects(objs, segs, seq.instance_class, visible)
        if gt_vox is not None:
            m.update(semantic_miou(objs, gt_vox, classes))
        rows[name] = m
    return rows


def _rows_to_df(rows, **extra):
    recs = []
    for name, m in rows.items():
        r = {"method": name, **extra}
        r.update({k: m.get(k, np.nan) for k in KEYS})
        recs.append(r)
    return pd.DataFrame(recs)


def fusion_study(seq, cache, frames, classes, pose_sets, seg_kwargs=None, max_depth=8.0, visible_kw=None):
    """pose_sets: {name: (poses, sigmas)} e.g. {'ground-truth poses': (gt, None), 'SLAM poses': (est, sig)}"""
    visible = visible_gt_assets(seq, frames, max_depth=max_depth, **(visible_kw or {}))
    gt_vox = gt_voxel_labels(seq, frames, seq.instance_class, classes, max_depth=max_depth, include_ids=visible)
    dfs = []
    for pname, (P, S) in pose_sets.items():
        by_frame = lift(seq, cache, frames, P, classes, S, seg_kwargs, max_depth)
        dfs.append(_rows_to_df(evaluate_all(seq, by_frame, frames, P, classes, visible, gt_vox), poses=pname))
    df = pd.concat(dfs, ignore_index=True)
    df.attrs["visible_assets"] = len(visible)
    return df, visible


def perturb_poses(poses, sigma_t, seed=0, drift=True):
    """Pose error model. drift=True: random-walk drift (how SLAM error behaves); False: per-frame jitter.
    sigma_t is the RMS translation error reached at the end of the sequence (metres); rotation
    error scales as 2 degrees per 0.1 m."""
    rng = np.random.default_rng(seed)
    n = len(poses)
    from scipy.spatial.transform import Rotation
    if drift:
        steps_t = rng.normal(0, sigma_t / np.sqrt(n), (n, 3)); steps_r = rng.normal(0, np.radians(20 * sigma_t) / np.sqrt(n), (n, 3))
        dt, dr = np.cumsum(steps_t, 0), np.cumsum(steps_r, 0)
    else:
        dt = rng.normal(0, sigma_t, (n, 3)); dr = rng.normal(0, np.radians(20 * sigma_t), (n, 3))
    out = []
    for T, t, r in zip(poses, dt, dr):
        E = np.eye(4); E[:3, :3] = Rotation.from_rotvec(r).as_matrix(); E[:3, 3] = t
        out.append(T @ E)
    return out


def pose_noise_study(seq, cache, frames, classes, levels=(0.0, 0.05, 0.1, 0.2, 0.4), seeds=(0, 1),
                     methods=("GASF (ours)", "Centroid tracking", "Geometry-only clustering", "Independent 2D labelling"),
                     seg_kwargs=None, max_depth=8.0, drift=True):
    visible = visible_gt_assets(seq, frames, max_depth=max_depth)
    dfs = []
    for lv in levels:
        for sd in (seeds if lv > 0 else (0,)):
            P = perturb_poses(seq.gt_poses, lv, sd, drift) if lv > 0 else seq.gt_poses
            by_frame = lift(seq, cache, frames, P, classes, None, seg_kwargs, max_depth)
            dfs.append(_rows_to_df(evaluate_all(seq, by_frame, frames, P, classes, visible, None, list(methods)),
                                   pose_error_m=lv, seed=sd))
    df = pd.concat(dfs, ignore_index=True)
    return df.groupby(["pose_error_m", "method"], as_index=False).mean(numeric_only=True)


def threshold_study(seq, cache, frames, classes, thresholds=(0.25, 0.30, 0.35, 0.40, 0.45),
                    methods=("GASF (ours)", "Centroid tracking", "Independent 2D labelling"), seg_kwargs=None,
                    max_depth=8.0):
    visible = visible_gt_assets(seq, frames, max_depth=max_depth)
    dfs = []
    for th in thresholds:
        kw = dict(seg_kwargs or {}, min_score=th)
        by_frame = lift(seq, cache, frames, seq.gt_poses, classes, None, kw, max_depth)
        dfs.append(_rows_to_df(evaluate_all(seq, by_frame, frames, seq.gt_poses, classes, visible, None,
                                            list(methods)), threshold=th))
    return pd.concat(dfs, ignore_index=True)


# ---------------------------------------------------------------- SLAM
SLAM_CONFIGS = {
    "Full (hybrid + PnP + loops)": dict(tracker="hybrid"),
    "w/o PnP recovery": dict(tracker="hybrid", use_pnp=False),
    "ICP tracker (geometry only)": dict(tracker="icp"),
}
# derived for free from the runs above (pose graph not optimised)
DERIVED = {"w/o loop closure": "Full (hybrid + PnP + loops)", "Odometry only (no PnP, no loops)": "w/o PnP recovery"}


def _slam_row(sname, cname, P, seq, extra):
    a, r = ate(P, seq.gt_poses), rpe(P, seq.gt_poses, seq.timestamps)
    return dict(sequence=sname, method=cname, ate_m=a["rmse"], rpe_m=r["trans_rmse"], rpe_deg=r["rot_rmse_deg"],
                path_m=path_length(seq.gt_poses), **extra)


def slam_study(sequences, configs=None, depth_max=4.0, verbose=True, derived=True, keep_poses=False):
    """sequences: {name: seq}. Returns (DataFrame with ATE/RPE per sequence x configuration, poses dict)."""
    rows, poses = [], {}
    for sname, seq in sequences.items():
        runs, ms = {}, {}
        for cname, kw in (configs or SLAM_CONFIGS).items():
            t = time.time()
            slam, P = run_slam(seq, depth_max=kw.get("depth_max", depth_max), verbose=False,
                               **{k: v for k, v in kw.items() if k != "depth_max"})
            el = time.time() - t
            runs[cname] = slam; ms[cname] = 1000 * el / len(seq)
            poses[(sname, cname)] = (P, slam.step_sigmas)
            rows.append(_slam_row(sname, cname, P, seq, dict(loops=slam.n_loops, fallbacks=slam.n_track_fail,
                                                              ms_per_frame=1000 * el / len(seq))))
            if verbose:
                print(f"  {sname:45s} {cname:32s} ATE {100 * rows[-1]['ate_m']:.1f} cm  ({el / 60:.1f} min)")
        if derived:
            for dname, src in DERIVED.items():
                if src in runs:
                    rows.append(_slam_row(sname, dname, runs[src].poses_before_opt, seq,
                                          dict(loops=0, fallbacks=runs[src].n_track_fail,
                                               ms_per_frame=ms[src])))
        for m, v in PUBLISHED_ATE.get(sname, {}).items():
            rows.append(dict(sequence=sname, method=f"{m} (published)", ate_m=v))
    df = pd.DataFrame(rows)
    return (df, poses) if keep_poses else df


# ---------------------------------------------------------------- plots & tables
def _style(ax, title, ylabel=None):
    ax.set_title(title, loc="left", fontsize=12, color="#1d2433")
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.spines["left"].set_color("#c9ced8"); ax.spines["bottom"].set_color("#c9ced8")
    ax.tick_params(colors="#5b6577"); ax.grid(axis="y", color="#e6e9ef", lw=0.8); ax.set_axisbelow(True)
    if ylabel:
        ax.set_ylabel(ylabel, color="#5b6577")


def plot_methods(df, path, metrics=("precision", "recall", "f1", "identity_consistency", "miou"),
                 methods=("GASF (ours)", "Centroid tracking", "Geometry-only clustering", "Independent 2D labelling"),
                 title="Semantic fusion vs conventional methods"):
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    d = df.groupby("method").mean(numeric_only=True)
    methods = [m for m in methods if m in d.index]
    metrics = [m for m in metrics if m in d.columns and d[m].notna().any()]
    fig, ax = plt.subplots(figsize=(9, 4.2))
    w = 0.8 / len(methods); x = np.arange(len(metrics))
    for k, m in enumerate(methods):
        v = d.loc[m, list(metrics)].values.astype(float)
        bars = ax.bar(x + k * w - 0.4 + w / 2, v, w * 0.92, color=PALETTE.get(m, "#888"), label=m)
        if m == "GASF (ours)":
            for b, val in zip(bars, v):
                ax.text(b.get_x() + b.get_width() / 2, val + 0.015, f"{val:.2f}", ha="center", fontsize=8,
                        color="#1d2433", weight="bold")
    ax.set_xticks(x, [m.replace("_", " ") for m in metrics]); ax.set_ylim(0, 1.08)
    _style(ax, title); ax.legend(frameon=False, ncol=4, fontsize=9, loc="upper center", bbox_to_anchor=(0.5, -0.1))
    fig.tight_layout(); fig.savefig(path, dpi=160, bbox_inches="tight"); plt.close(fig)


def plot_ablation(df, path, metric="f1", title="GASF ablation (F1)"):
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    d = df.groupby("method").mean(numeric_only=True)
    order = [m for m in METHOD_ORDER if m in d.index]
    fig, ax = plt.subplots(figsize=(7.5, 3.6))
    vals = [d.loc[m, metric] for m in order]
    ax.barh(order[::-1], vals[::-1], color=[PALETTE.get(m, "#888") for m in order[::-1]])
    for y, v in enumerate(vals[::-1]):
        ax.text(v + 0.01, y, f"{v:.2f}", va="center", fontsize=9, color="#1d2433")
    ax.set_xlim(0, 1.05); _style(ax, title); ax.grid(axis="x", color="#e6e9ef"); ax.grid(axis="y", visible=False)
    fig.tight_layout(); fig.savefig(path, dpi=160, bbox_inches="tight"); plt.close(fig)


def plot_lines(df, x, y, path, title, xlabel, ylabel, methods=None):
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6.4, 3.8))
    ms = methods or [m for m in METHOD_ORDER if m in set(df["method"])]
    for m in sorted(ms, key=lambda m: m == "GASF (ours)"):            # draw ours last so it stays visible
        d = df[df["method"] == m].sort_values(x)
        if len(d):
            ours = m == "GASF (ours)"
            ax.plot(d[x], d[y], marker="o", ms=6 if ours else 4, lw=2.6 if ours else 1.6, zorder=3 if ours else 2,
                    color=PALETTE.get(m, "#888"), label=m, ls="-" if ours else "--")
    ax.set_xlabel(xlabel, color="#5b6577"); _style(ax, title, ylabel); ax.legend(frameon=False, fontsize=9)
    fig.tight_layout(); fig.savefig(path, dpi=160, bbox_inches="tight"); plt.close(fig)


def plot_slam(df, path, title="Trajectory error (ATE RMSE)"):
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    seqs = list(dict.fromkeys(df["sequence"]))
    fig, axes = plt.subplots(1, len(seqs), figsize=(4.6 * len(seqs), 3.8), squeeze=False)
    for ax, s in zip(axes[0], seqs):
        d = df[df["sequence"] == s].dropna(subset=["ate_m"])
        cols = ["#9aa3b2" if "published" in m else ("#1F5BD8" if m.startswith("Full") else "#7FA3EA") for m in d["method"]]
        ax.barh(d["method"][::-1], 100 * d["ate_m"][::-1], color=cols[::-1])
        for yy, v in enumerate(100 * d["ate_m"][::-1]):
            ax.text(v, yy, f" {v:.1f}", va="center", fontsize=8)
        _style(ax, s.replace("rgbd_dataset_", "")); ax.set_xlabel("ATE (cm)", color="#5b6577")
        ax.grid(axis="x", color="#e6e9ef"); ax.grid(axis="y", visible=False)
    fig.tight_layout(); fig.savefig(path, dpi=160, bbox_inches="tight"); plt.close(fig)


def table_md(df, cols=None, floatfmt=3):
    d = df[cols] if cols else df
    try:
        return d.to_markdown(index=False, floatfmt=f".{floatfmt}f")
    except ImportError:
        return d.round(floatfmt).to_string(index=False)


def save_study(outdir, name, df):
    outdir = Path(outdir); outdir.mkdir(parents=True, exist_ok=True)
    df.to_csv(outdir / f"{name}.csv", index=False)
    (outdir / f"{name}.json").write_text(df.to_json(orient="records", indent=1))
    return outdir / f"{name}.csv"

In [ ]:
%%writefile /content/geosemtwinx/dashboard.py
"""Digital-twin dashboard: one self-contained HTML file (Plotly from CDN, data embedded).

Sections: KPIs · interactive 3D twin (map, assets, trajectories) · asset inspector + spatial query ·
filterable asset table · change log + version history · comparative studies · system performance.
"""
import base64
import datetime as _dt
import json
from pathlib import Path

import numpy as np

CLASS_COLORS = ["#1F5BD8", "#D9731A", "#2A9D7F", "#A04FB0", "#C23B5A", "#5B7F1F", "#8A6D3B", "#3A8FB7"]


def _img(path):
    p = Path(path)
    if not p.exists():
        return None
    return "data:image/png;base64," + base64.b64encode(p.read_bytes()).decode()


def _df_table(df, floatfmt=3):
    d = df.copy()
    for c in d.columns:
        if d[c].dtype.kind == "f":
            d[c] = d[c].round(floatfmt)
    return dict(columns=[str(c) for c in d.columns], rows=d.astype(object).where(d.notna(), None).values.tolist())


def study(title, note="", df=None, image=None, highlight="GASF (ours)"):
    """Helper to describe one comparative-study card."""
    return dict(title=title, note=note, table=_df_table(df) if df is not None else None, image=_img(image) if image else None,
                highlight=highlight)


def build_dashboard(path, twin, map_points=None, map_colors=None, traj_est=None, traj_gt=None, studies=(),
                    perf=None, title="GeoSemTwin-X Digital Twin", subtitle="", max_points=40000, inline_plotly=True):
    """inline_plotly=True embeds plotly.js (~4.6 MB) so the file also works offline / after download."""
    tw = twin.to_dict() if hasattr(twin, "to_dict") else twin
    pts = None
    if map_points is not None and len(map_points):
        P = np.asarray(map_points); idx = np.random.default_rng(0).choice(len(P), min(max_points, len(P)), replace=False)
        P = P[idx]
        C = (np.asarray(map_colors)[idx] * 255).astype(int) if map_colors is not None else np.full((len(P), 3), 150)
        pts = dict(x=np.round(P[:, 0], 3).tolist(), y=np.round(P[:, 1], 3).tolist(), z=np.round(P[:, 2], 3).tolist(),
                   c=[f"rgb({r},{g},{b})" for r, g, b in C])
    labels = sorted({a["label"] for a in tw["assets"]})
    data = dict(title=title, subtitle=subtitle, generated=_dt.datetime.now().strftime("%Y-%m-%d %H:%M"), twin=tw,
                map=pts, traj_est=np.round(np.asarray(traj_est), 3).tolist() if traj_est is not None else None,
                traj_gt=np.round(np.asarray(traj_gt), 3).tolist() if traj_gt is not None else None,
                colors={l: CLASS_COLORS[i % len(CLASS_COLORS)] for i, l in enumerate(labels)},
                studies=list(studies), perf=perf or {})
    html = TEMPLATE.replace("/*__DATA__*/", json.dumps(data, default=float).replace("</", "<\\/"))
    if inline_plotly:
        try:
            import plotly
            js = (Path(plotly.__file__).parent / "package_data" / "plotly.min.js").read_text(encoding="utf-8")
            html = html.replace('<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>',
                                "<script>" + js.replace("</script", "<\\/script") + "</script>")
        except Exception:
            pass
    Path(path).write_text(html, encoding="utf-8")
    return path


TEMPLATE = r"""<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>GeoSemTwin-X Dashboard</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
:root{--bg:#f4f6f9;--card:#ffffff;--ink:#1d2433;--muted:#5b6577;--line:#e3e7ee;--accent:#1F5BD8;--accent-soft:#eaf0fc;
--good:#1f7a3a;--warn:#b45309;--bad:#b4232a;--shadow:0 1px 2px rgba(16,24,40,.06),0 1px 3px rgba(16,24,40,.08)}
@media (prefers-color-scheme: dark){:root:not([data-theme="light"]){--bg:#0f141c;--card:#171e29;--ink:#e6eaf2;--muted:#9aa5b8;
--line:#263142;--accent:#6d9cff;--accent-soft:#1c2a45;--shadow:none}}
:root[data-theme="dark"]{--bg:#0f141c;--card:#171e29;--ink:#e6eaf2;--muted:#9aa5b8;--line:#263142;--accent:#6d9cff;--accent-soft:#1c2a45;--shadow:none}
*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--ink);font:14px/1.45 system-ui,-apple-system,"Segoe UI",Roboto,Arial,sans-serif}
header{display:flex;justify-content:space-between;align-items:flex-end;gap:16px;padding:20px 24px 8px;flex-wrap:wrap}
h1{font-size:22px;margin:0}h2{font-size:15px;margin:0 0 10px}.sub{color:var(--muted);font-size:13px}
nav{display:flex;gap:6px;padding:0 24px 12px;flex-wrap:wrap}nav button{border:1px solid var(--line);background:var(--card);color:var(--ink);
padding:7px 14px;border-radius:999px;cursor:pointer;font:inherit}nav button.on{background:var(--accent);border-color:var(--accent);color:#fff}
main{padding:0 24px 32px}.grid{display:grid;gap:14px}.kpis{grid-template-columns:repeat(auto-fit,minmax(150px,1fr));margin-bottom:14px}
.card{background:var(--card);border:1px solid var(--line);border-radius:12px;padding:14px 16px;box-shadow:var(--shadow);min-width:0}
.kpi .v{font-size:26px;font-weight:700;letter-spacing:-.01em}.kpi .l{color:var(--muted);font-size:12px}
.two{grid-template-columns:minmax(0,2.1fr) minmax(280px,1fr)}@media(max-width:900px){.two{grid-template-columns:1fr}}
#plot3d{height:560px}.chips{display:flex;flex-wrap:wrap;gap:6px;margin-bottom:8px}.chip{display:inline-flex;align-items:center;gap:6px;
border:1px solid var(--line);border-radius:999px;padding:3px 10px;font-size:12px;cursor:pointer;user-select:none;background:var(--card)}
.chip.off{opacity:.4}.dot{width:9px;height:9px;border-radius:50%}
table{width:100%;border-collapse:collapse;font-size:13px}th,td{padding:7px 8px;border-bottom:1px solid var(--line);text-align:left;white-space:nowrap}
th{color:var(--muted);font-weight:600;cursor:pointer;position:sticky;top:0;background:var(--card)}td.num,th.num{text-align:right;font-variant-numeric:tabular-nums}
tr.sel td{background:var(--accent-soft)}tbody tr:hover td{background:var(--accent-soft);cursor:pointer}tr.hl td{font-weight:700}
.scroll{overflow:auto;max-height:420px}.row{display:flex;gap:8px;align-items:center;flex-wrap:wrap;margin-bottom:8px}
input,select{font:inherit;color:var(--ink);background:var(--bg);border:1px solid var(--line);border-radius:8px;padding:6px 8px;min-width:0}
input[type=number]{width:84px}button.act{background:var(--accent);color:#fff;border:0;border-radius:8px;padding:7px 12px;cursor:pointer;font:inherit}
.kv{display:grid;grid-template-columns:auto 1fr;gap:4px 12px;font-size:13px}.kv div:nth-child(odd){color:var(--muted)}
.badge{display:inline-block;padding:1px 8px;border-radius:999px;font-size:12px;border:1px solid var(--line)}
.moved{color:var(--warn)}.removed{color:var(--bad)}.added{color:var(--good)}.empty{color:var(--muted);padding:12px 0}
.study img{width:100%;height:auto;border-radius:8px;border:1px solid var(--line);background:#fff}.note{color:var(--muted);font-size:12px;margin:-4px 0 10px}
.studies{grid-template-columns:repeat(auto-fit,minmax(420px,1fr))}@media(max-width:520px){.studies{grid-template-columns:1fr}#plot3d{height:420px}}
.tab{display:none}.tab.on{display:block}.bar{height:8px;border-radius:4px;background:var(--accent)}
</style></head><body>
<header><div><h1 id="title"></h1><div class="sub" id="subtitle"></div></div>
<div class="row"><span class="sub" id="gen"></span><button class="chip" id="theme" title="Toggle theme">◐ Theme</button></div></header>
<nav id="tabs"><button data-t="twin" class="on">Digital twin</button><button data-t="changes">Changes &amp; versions</button>
<button data-t="studies">Comparative studies</button><button data-t="perf">System performance</button></nav>
<main>
<section class="grid kpis" id="kpis"></section>
<div class="tab on" id="tab-twin">
 <div class="grid two">
  <div class="card"><div class="row" style="justify-content:space-between"><h2 style="margin:0">3D twin</h2>
   <label class="sub"><input type="checkbox" id="showmap" checked> map points</label></div>
   <div class="chips" id="classchips"></div><div id="plot3d"></div></div>
  <div class="grid" style="align-content:start">
   <div class="card"><h2>Asset inspector</h2><div id="inspector" class="empty">Click an asset in the table or the 3D view.</div></div>
   <div class="card"><h2>Spatial query</h2>
    <div class="row">x <input type="number" id="qx" step="0.5"> y <input type="number" id="qy" step="0.5"> r <input type="number" id="qr" value="5" step="0.5"></div>
    <div class="row"><select id="qc"></select><button class="act" id="qbtn">Find assets</button></div><div id="qres" class="sub"></div></div>
  </div></div>
 <div class="card" style="margin-top:14px"><div class="row" style="justify-content:space-between"><h2 style="margin:0">Asset registry</h2>
  <div class="row"><input id="search" placeholder="Search ID or class"><select id="fclass"></select>
  <label class="sub">min confidence <input type="number" id="fconf" value="0" min="0" max="1" step="0.05"></label></div></div>
  <div class="scroll"><table id="assets"></table></div></div>
</div>
<div class="tab" id="tab-changes"><div class="grid two">
 <div class="card"><h2>Change log</h2><div class="scroll"><table id="changes"></table></div></div>
 <div class="card"><h2>Version history</h2><div class="scroll"><table id="versions"></table></div></div></div></div>
<div class="tab" id="tab-studies"><div class="grid studies" id="studies"></div></div>
<div class="tab" id="tab-perf"><div class="grid two"><div class="card"><h2>Per-stage latency</h2><div id="perfbars"></div></div>
 <div class="card"><h2>Run details</h2><div class="kv" id="perfkv"></div></div></div></div>
</main>
<script>
const D=/*__DATA__*/;
const $=s=>document.querySelector(s), esc=s=>String(s??"").replace(/[&<>"]/g,c=>({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;"}[c]));
const A=D.twin.assets.filter(a=>a.status!=="removed"), COL=D.colors;
$("#title").textContent=D.title; $("#subtitle").textContent=D.subtitle||D.twin.name; $("#gen").textContent="Generated "+D.generated+" · twin v"+D.twin.version;
document.querySelectorAll("#tabs button").forEach(b=>b.onclick=()=>{document.querySelectorAll("#tabs button").forEach(x=>x.classList.toggle("on",x===b));
 document.querySelectorAll(".tab").forEach(t=>t.classList.toggle("on",t.id==="tab-"+b.dataset.t)); if(b.dataset.t==="twin")Plotly.Plots.resize("plot3d");});
$("#theme").onclick=()=>{const r=document.documentElement;const dark=r.dataset.theme?r.dataset.theme==="dark":matchMedia("(prefers-color-scheme: dark)").matches;r.dataset.theme=dark?"light":"dark";draw();};
// KPIs
const classes=[...new Set(A.map(a=>a.label))].sort(), obs=A.reduce((s,a)=>s+(a.observations||0),0);
const meanConf=A.length?A.reduce((s,a)=>s+(a.confidence||0),0)/A.length:0;
const kp=[["Assets in twin",A.length],["Asset classes",classes.length],["Observations fused",obs],["Mean label confidence",meanConf.toFixed(2)],
 ["Changes logged",(D.twin.changes||[]).length],["Twin versions",(D.twin.history||[]).length]];
if(D.perf.ate_cm!=null)kp.push(["Trajectory error (ATE)",D.perf.ate_cm.toFixed(1)+" cm"]);
$("#kpis").innerHTML=kp.map(([l,v])=>`<div class="card kpi"><div class="v">${esc(v)}</div><div class="l">${esc(l)}</div></div>`).join("");
// class chips + filters
const hidden=new Set();
$("#classchips").innerHTML=classes.map(c=>`<span class="chip" data-c="${esc(c)}"><span class="dot" style="background:${COL[c]}"></span>${esc(c)} (${A.filter(a=>a.label===c).length})</span>`).join("");
document.querySelectorAll("#classchips .chip").forEach(ch=>ch.onclick=()=>{const c=ch.dataset.c;hidden.has(c)?hidden.delete(c):hidden.add(c);ch.classList.toggle("off");draw();});
$("#fclass").innerHTML=`<option value="">All classes</option>`+classes.map(c=>`<option>${esc(c)}</option>`).join("");
$("#qc").innerHTML=`<option value="">Any class</option>`+classes.map(c=>`<option>${esc(c)}</option>`).join("");
let selected=null, query=null;
function boxLines(a){const[x0,y0,z0]=a.bbox_min,[x1,y1,z1]=a.bbox_max;const c=[[x0,y0,z0],[x1,y0,z0],[x1,y1,z0],[x0,y1,z0],[x0,y0,z1],[x1,y0,z1],[x1,y1,z1],[x0,y1,z1]];
 const e=[[0,1],[1,2],[2,3],[3,0],[4,5],[5,6],[6,7],[7,4],[0,4],[1,5],[2,6],[3,7]];const X=[],Y=[],Z=[];e.forEach(([p,q])=>{X.push(c[p][0],c[q][0],null);Y.push(c[p][1],c[q][1],null);Z.push(c[p][2],c[q][2],null)});return[X,Y,Z];}
function draw(){const dark=getComputedStyle(document.documentElement).getPropertyValue("--bg").trim()!=="#f4f6f9";const ink=getComputedStyle(document.documentElement).getPropertyValue("--ink").trim();
 const tr=[];
 if(D.map&&$("#showmap").checked)tr.push({type:"scatter3d",mode:"markers",x:D.map.x,y:D.map.y,z:D.map.z,marker:{size:1.4,color:D.map.c,opacity:.55},hoverinfo:"skip",name:"map",showlegend:false});
 classes.filter(c=>!hidden.has(c)).forEach(c=>{const L=A.filter(a=>a.label===c);const X=[],Y=[],Z=[];L.forEach(a=>{const[bx,by,bz]=boxLines(a);X.push(...bx);Y.push(...by);Z.push(...bz)});
  tr.push({type:"scatter3d",mode:"lines",x:X,y:Y,z:Z,line:{color:COL[c],width:4},hoverinfo:"skip",showlegend:false});
  tr.push({type:"scatter3d",mode:"markers",x:L.map(a=>a.centroid[0]),y:L.map(a=>a.centroid[1]),z:L.map(a=>a.bbox_max[2]+.1),customdata:L.map(a=>a.id),
   marker:{size:5,color:COL[c],line:{color:"#fff",width:1}},name:c,text:L.map(a=>`${a.id} · ${a.label}<br>conf ${(+a.confidence).toFixed(2)} · obs ${a.observations}<br>size ${a.size.map(v=>(+v).toFixed(2)).join(" × ")} m`),hoverinfo:"text"});});
 if(D.traj_gt)tr.push({type:"scatter3d",mode:"lines",x:D.traj_gt.map(p=>p[0]),y:D.traj_gt.map(p=>p[1]),z:D.traj_gt.map(p=>p[2]),line:{color:"#2a9d8f",width:3},name:"ground-truth path"});
 if(D.traj_est)tr.push({type:"scatter3d",mode:"lines",x:D.traj_est.map(p=>p[0]),y:D.traj_est.map(p=>p[1]),z:D.traj_est.map(p=>p[2]),line:{color:"#e76f51",width:3},name:"estimated path"});
 if(selected){const[bx,by,bz]=boxLines(selected);tr.push({type:"scatter3d",mode:"lines",x:bx,y:by,z:bz,line:{color:"#e11d48",width:9},hoverinfo:"skip",name:"selected "+selected.id});}
 if(query){const t=[...Array(65).keys()].map(i=>2*Math.PI*i/64);tr.push({type:"scatter3d",mode:"lines",x:t.map(a=>query.x+query.r*Math.cos(a)),y:t.map(a=>query.y+query.r*Math.sin(a)),z:t.map(()=>query.z),line:{color:"#f59e0b",width:5,dash:"dash"},name:"query radius",hoverinfo:"skip"});}
 Plotly.react("plot3d",tr,{margin:{l:0,r:0,t:0,b:0},paper_bgcolor:"rgba(0,0,0,0)",font:{color:ink},legend:{orientation:"h",y:1.02,x:0},
  scene:{aspectmode:"data",xaxis:{title:"x (m)",color:ink},yaxis:{title:"y (m)",color:ink},zaxis:{title:"z (m)",color:ink}}},{displaylogo:false,responsive:true});
 const el=$("#plot3d");el.removeAllListeners&&el.removeAllListeners("plotly_click");el.on("plotly_click",e=>{const id=e.points[0].customdata;if(id)select(id);});}
function select(id){selected=A.find(a=>a.id===id);const a=selected;if(!a)return;
 $("#inspector").className="";$("#inspector").innerHTML=`<div class="row"><span class="dot" style="background:${COL[a.label]}"></span><b>${esc(a.id)}</b> <span class="badge">${esc(a.label)}</span></div>
 <div class="kv"><div>Confidence</div><div>${(+a.confidence).toFixed(3)} (entropy ${(+a.label_entropy).toFixed(2)})</div><div>Centroid</div><div>${a.centroid.map(v=>(+v).toFixed(2)).join(", ")} m</div>
 <div>Size</div><div>${a.size.map(v=>(+v).toFixed(2)).join(" × ")} m</div><div>Observations</div><div>${a.observations} from ${a.views} viewpoints</div>
 <div>First / last seen</div><div>${esc(a.first_seen)}<br>${esc(a.last_seen)}</div><div>Versions</div><div>${(a.versions||[]).map(v=>"v"+v.version).join(", ")}</div><div>Status</div><div>${esc(a.status)}</div></div>`;
 renderTable();draw();}
// asset table
let sortKey="id",sortDir=1;
const cols=[["id","ID"],["label","Class"],["confidence","Conf.",1],["observations","Obs.",1],["views","Views",1],["cx","x (m)",1],["cy","y (m)",1],["cz","z (m)",1],["vol","Size (m³)",1]];
function renderTable(){const q=$("#search").value.toLowerCase(),fc=$("#fclass").value,mc=+$("#fconf").value||0;
 let R=A.filter(a=>(!fc||a.label===fc)&&(+a.confidence>=mc)&&(!q||a.id.toLowerCase().includes(q)||a.label.toLowerCase().includes(q)))
  .map(a=>({...a,cx:a.centroid[0],cy:a.centroid[1],cz:a.centroid[2],vol:a.size[0]*a.size[1]*a.size[2]}));
 R.sort((p,q2)=>(p[sortKey]>q2[sortKey]?1:p[sortKey]<q2[sortKey]?-1:0)*sortDir);
 $("#assets").innerHTML=`<thead><tr>${cols.map(([k,l,n])=>`<th data-k="${k}" class="${n?"num":""}">${l}${sortKey===k?(sortDir>0?" ▲":" ▼"):""}</th>`).join("")}</tr></thead><tbody>`+
  (R.length?R.map(a=>`<tr data-id="${esc(a.id)}" class="${selected&&selected.id===a.id?"sel":""}">${cols.map(([k,,n])=>`<td class="${n?"num":""}">${k==="label"?`<span class="dot" style="display:inline-block;background:${COL[a.label]}"></span> ${esc(a.label)}`:n?(+a[k]).toFixed(k==="observations"||k==="views"?0:2):esc(a[k])}</td>`).join("")}</tr>`).join(""):`<tr><td colspan="${cols.length}" class="empty">No assets match the filters.</td></tr>`)+"</tbody>";
 document.querySelectorAll("#assets th").forEach(th=>th.onclick=()=>{const k=th.dataset.k;sortDir=sortKey===k?-sortDir:1;sortKey=k;renderTable();});
 document.querySelectorAll("#assets tbody tr[data-id]").forEach(tr=>tr.onclick=()=>select(tr.dataset.id));}
["#search","#fclass","#fconf"].forEach(s=>$(s).addEventListener("input",renderTable));
$("#showmap").onchange=draw;
// spatial query
const cx=A.length?A.reduce((s,a)=>s+a.centroid[0],0)/A.length:0, cy=A.length?A.reduce((s,a)=>s+a.centroid[1],0)/A.length:0;
$("#qx").value=cx.toFixed(1);$("#qy").value=cy.toFixed(1);
$("#qbtn").onclick=()=>{const x=+$("#qx").value,y=+$("#qy").value,r=+$("#qr").value,c=$("#qc").value;
 const z=A.length?Math.min(...A.map(a=>a.bbox_min[2])):0;query={x,y,r,z};
 const hits=A.filter(a=>(!c||a.label===c)).map(a=>({a,d:Math.hypot(a.centroid[0]-x,a.centroid[1]-y)})).filter(h=>h.d<=r).sort((p,q)=>p.d-q.d);
 $("#qres").innerHTML=hits.length?`${hits.length} asset(s) within ${r} m:<br>`+hits.map(h=>`<a href="#" data-id="${esc(h.a.id)}">${esc(h.a.id)}</a> ${esc(h.a.label)} · ${h.d.toFixed(1)} m`).join("<br>"):"No assets in this radius.";
 document.querySelectorAll("#qres a").forEach(l=>l.onclick=e=>{e.preventDefault();select(l.dataset.id);});draw();};
// changes + versions
const ch=D.twin.changes||[];
$("#changes").innerHTML=ch.length?`<thead><tr><th>Version</th><th>Type</th><th>Class</th><th>Asset</th><th class="num">Detail</th></tr></thead><tbody>`+ch.map(c=>`<tr><td>v${c.version}</td><td class="${esc(c.type)}"><b>${esc(c.type)}</b></td><td>${esc(c.label)}</td><td>${esc(c.b||c.a||"")}</td><td class="num">${c.displacement!=null?(+c.displacement).toFixed(2)+" m":c.chamfer!=null?"Δshape "+(+c.chamfer).toFixed(2)+" m":""}</td></tr>`).join("")+"</tbody>"
 :`<tbody><tr><td class="empty">No changes logged yet. Commit a second mapping session to compare versions.</td></tr></tbody>`;
const hv=D.twin.history||[];
$("#versions").innerHTML=`<thead><tr><th>Version</th><th>Session</th><th>Time</th><th class="num">Assets</th></tr></thead><tbody>`+hv.map(h=>`<tr><td>v${h.version}</td><td>${esc(h.session)}</td><td>${esc(h.time)}</td><td class="num">${h.n_assets}</td></tr>`).join("")+"</tbody>";
// studies
$("#studies").innerHTML=D.studies.length?D.studies.map(s=>`<div class="card study"><h2>${esc(s.title)}</h2>${s.note?`<div class="note">${esc(s.note)}</div>`:""}
 ${s.image?`<img src="${s.image}" alt="${esc(s.title)} chart">`:""}${s.table?`<div class="scroll" style="margin-top:10px;max-height:320px"><table><thead><tr>${s.table.columns.map(c=>`<th>${esc(c)}</th>`).join("")}</tr></thead><tbody>${s.table.rows.map(r=>`<tr class="${r.includes(s.highlight)?"hl":""}">${r.map(v=>`<td class="${typeof v==="number"?"num":""}">${v===null?"—":esc(v)}</td>`).join("")}</tr>`).join("")}</tbody></table></div>`:""}</div>`).join("")
 :`<div class="card empty">No comparative studies attached to this dashboard.</div>`;
// performance
const st=D.perf.stages_ms||{};const mx=Math.max(1,...Object.values(st));
$("#perfbars").innerHTML=Object.keys(st).length?Object.entries(st).map(([k,v])=>`<div style="margin-bottom:10px"><div class="row" style="justify-content:space-between;margin:0"><span>${esc(k)}</span><span class="sub">${(+v).toFixed(0)} ms</span></div><div class="bar" style="width:${Math.max(2,100*v/mx)}%"></div></div>`).join(""):`<div class="empty">No timing data.</div>`;
$("#perfkv").innerHTML=Object.entries(D.perf.details||{}).map(([k,v])=>`<div>${esc(k)}</div><div>${esc(v)}</div>`).join("")||`<div class="empty">—</div>`;
renderTable();draw();
</script></body></html>"""

In [ ]:
import sys, os, json, time
sys.path.insert(0, "/content")
import numpy as np, pandas as pd
from IPython.display import HTML, Image, Markdown, display
import geosemtwinx
from geosemtwinx import compare as CMP
from geosemtwinx.data import TUMSequence, StaerSequence
from geosemtwinx.slam import run_slam
from geosemtwinx.metrics import ate, rpe, path_length
from geosemtwinx.pipeline import run_mapping, visible_gt_assets
from geosemtwinx.twin import DigitalTwin, TwinStore
from geosemtwinx.dashboard import build_dashboard, study
ck = "/content/checkpoints/sam2.1_hiera_tiny.pt"
if not os.path.exists(ck):
    !wget -q -O {ck} https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt
OUT, CMPD, FIGD = "/content/results", "/content/results/comparative", "/content/results/figures"
for d in (OUT, CMPD, FIGD):
    os.makedirs(d, exist_ok=True)
RESULTS = {"env": {"gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "torch": torch.__version__}}
STUDIES = []          # cards for the dashboard
def save_results():
    json.dump(RESULTS, open(f"{OUT}/results_colab.json", "w"), indent=1, default=float)
def show(df, cols=None, n=3):
    display(Markdown(CMP.table_md(df, cols, n)))

## 2 · Configuration

In [ ]:
RUN_TUM, RUN_STAER, RUN_SYNTHETIC = True, True, False
# TUM sequences for the SLAM comparison (published baselines exist for all three).
# fr1/desk takes ~10 min per configuration; the others are longer (fr2/xyz 3.6k frames, fr3/office 2.5k frames).
TUM_SEQS = ["rgbd_dataset_freiburg1_desk"]          # add "rgbd_dataset_freiburg2_xyz", "rgbd_dataset_freiburg3_long_office_household"
TUM_SLAM_CONFIGS = ["Full (hybrid + PnP + loops)", "w/o PnP recovery"]   # + "ICP tracker (geometry only)" (slow, ~30 min)
TUM_CLASSES = ["monitor", "keyboard", "computer mouse", "cup", "book", "chair", "telephone", "bottle"]
STAER_SCENE, STAER_WALK = "scene03", "001"
STAER_CLASSES = ["forklift", "pallet", "cardboard box", "pallet truck"]
FLOOR_CLASSES = ["forklift", "pallet truck"]        # must touch the floor (z = 0 in Staer world frame)
DET_CACHE_THRESHOLD = 0.20                          # detections are cached at a low threshold ...
SCORE_THRESHOLD = 0.30                              # ... and filtered here (threshold study varies this)
TWIN_STORE = f"{OUT}/twin_store"                    # e.g. "/content/drive/MyDrive/GeoSemTwinX_store" after mounting Drive

## 3 · Perception front end (Grounding DINO + SAM 2.1)

In [ ]:
from geosemtwinx.detectors import GroundedSAM2Detector
def make_detector(classes, box_threshold=DET_CACHE_THRESHOLD):
    return GroundedSAM2Detector(classes, sam_cfg="configs/sam2.1/sam2.1_hiera_t.yaml", sam_ckpt=ck,
                                box_threshold=box_threshold)

## Part A · TUM RGB-D
### A1 · SLAM comparison and component ablation
ATE / RPE computed as in the TUM benchmark. *w/o loop closure* and *Odometry only* come from the same runs before
pose-graph optimisation. Published numbers: ORB-SLAM2 paper, Table III.

In [ ]:
if RUN_TUM:
    tums = {s: TUMSequence(s, root="/content/data/tum") for s in TUM_SEQS}
    cfgs = {k: CMP.SLAM_CONFIGS[k] for k in TUM_SLAM_CONFIGS}
    df_slam, tum_poses = CMP.slam_study(tums, cfgs, depth_max=4.0, keep_poses=True)
    CMP.save_study(CMPD, "tum_slam", df_slam)
    show(df_slam, ["sequence", "method", "ate_m", "rpe_m", "rpe_deg", "loops", "ms_per_frame"])
    CMP.plot_slam(df_slam, f"{FIGD}/tum_slam.png")
    display(Image(f"{FIGD}/tum_slam.png"))
    STUDIES.append(study("SLAM accuracy on TUM RGB-D (ATE RMSE)",
        "Ours: full system and component ablations; grey: published RGB-D SLAM systems", df_slam[["sequence","method","ate_m","rpe_m","loops"]],
        f"{FIGD}/tum_slam.png", highlight="Full (hybrid + PnP + loops)"))
    RESULTS["tum_slam"] = df_slam.to_dict("records"); save_results()

In [ ]:
import matplotlib.pyplot as plt
if RUN_TUM:
    s0 = TUM_SEQS[0]; tum = tums[s0]; P_t, sig_t = tum_poses[(s0, "Full (hybrid + PnP + loops)")]
    a = ate(P_t, tum.gt_poses)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
    ax[0].plot(a["gt"][:, 0], a["gt"][:, 1], label="ground truth"); ax[0].plot(a["aligned"][:, 0], a["aligned"][:, 1], label="GeoSemTwin-X")
    ax[0].set_aspect("equal"); ax[0].legend(); ax[0].set_title(f"{s0}: trajectory (top view)")
    ax[1].plot(a["errors"] * 100); ax[1].set_ylabel("ATE (cm)"); ax[1].set_xlabel("frame"); ax[1].set_title("Absolute trajectory error")
    plt.tight_layout(); plt.savefig(f"{FIGD}/tum_trajectory.png", dpi=150); plt.show()

### A2 · Open-vocabulary office twin (no semantic ground truth in TUM → qualitative)

In [ ]:
if RUN_TUM:
    det_t = make_detector(TUM_CLASSES)
    frames_t = list(range(0, len(tum), 5))
    torch.cuda.reset_peak_memory_stats()
    cache_t, det_ms_t = CMP.cache_detections(tum, det_t, frames_t)
    poses_w = [a["T_align"] @ T for T in P_t]
    out_t = run_mapping(tum, poses_w, sig_t, None, TUM_CLASSES, gasf_stride=5, tsdf_voxel=0.01, tsdf_depth=3.0,
                        max_depth=4.0, use_gt_eval=False, det_cache=cache_t, seg_kwargs=dict(min_score=SCORE_THRESHOLD))
    twin_t = DigitalTwin("TUM fr1/desk twin"); twin_t.update(out_t["gasf"].finalize(), TUM_CLASSES, "tum-1", occ_new=out_t["occ"])
    pc = out_t["tsdf"].point_cloud()
    build_dashboard(f"{OUT}/TUM_Dashboard.html", twin_t, np.asarray(pc.points), np.asarray(pc.colors),
                    traj_est=[T[:3, 3] for T in poses_w], traj_gt=[T[:3, 3] for T in tum.gt_poses],
                    perf=dict(ate_cm=100 * a["rmse"], stages_ms={"Detector + SAM 2.1 (GPU)": det_ms_t, **out_t["timings_ms"]},
                              details={"GPU": RESULTS["env"]["gpu"], "Peak GPU memory": f"{torch.cuda.max_memory_allocated()/2**30:.1f} GB"}),
                    title="GeoSemTwin-X · TUM office twin", subtitle=s0)
    RESULTS["tum_semantic"] = dict(assets=twin_t.counts(), n_assets=len(twin_t.present()), detector_ms=det_ms_t,
                                   timings_ms=out_t["timings_ms"], map_points=len(pc.points))
    save_results(); print(json.dumps(RESULTS["tum_semantic"], indent=1))

## Part B · Staer warehouse (scene 03)
### B1 · Load data and SLAM configuration sweep
The v1 run drifted 1.5 m. The sweep tests tracker / depth-range settings on the first 120 frames and runs the
whole walkthrough with the best one.

In [ ]:
if RUN_STAER:
    from huggingface_hub import notebook_login
    notebook_login()

In [ ]:
if RUN_STAER:
    st = StaerSequence(STAER_SCENE, STAER_WALK, root="/content/data/staer", scale=0.5, stride=2)
    print("frames:", len(st), "| GT assets by class:", pd.Series(list(st.instance_class.values())).value_counts().to_dict())
    st_short = StaerSequence(STAER_SCENE, STAER_WALK, root="/content/data/staer", scale=0.5, stride=2, max_frames=60)
    sweep = {"hybrid, depth ≤ 10 m": dict(tracker="hybrid", depth_max=10.0),
             "hybrid, depth ≤ 6 m": dict(tracker="hybrid", depth_max=6.0),
             "hybrid, depth ≤ 6 m, stride 1": None,
             "hybrid + ICP, depth ≤ 8 m": dict(tracker="hybrid+icp", depth_max=8.0)}
    rows = []
    for name, kw in sweep.items():
        seq = st_short
        if kw is None:
            seq = StaerSequence(STAER_SCENE, STAER_WALK, root="/content/data/staer", scale=0.5, stride=1, max_frames=120)
            kw = dict(tracker="hybrid", depth_max=6.0)
        r = CMP.slam_study({"staer first 6 s": seq}, {name: kw}, derived=False, verbose=True)
        rows.append(r)
    df_sweep = pd.concat(rows, ignore_index=True); CMP.save_study(CMPD, "staer_slam_sweep", df_sweep)
    show(df_sweep, ["method", "ate_m", "rpe_m", "fallbacks", "ms_per_frame"])
    best = df_sweep.sort_values("ate_m").iloc[0]["method"]; print("best configuration:", best)

In [ ]:
if RUN_STAER:
    kw = sweep[best] or dict(tracker="hybrid", depth_max=6.0)
    st_slam = st if sweep[best] is not None else StaerSequence(STAER_SCENE, STAER_WALK, root="/content/data/staer", scale=0.5, stride=1)
    df_s, poses_s = CMP.slam_study({"staer scene03": st_slam}, {best: kw}, keep_poses=True)
    P_s, sig_s = poses_s[("staer scene03", best)]
    if st_slam is not st:                                   # map back to the stride-2 frames used below
        P_s, sig_s = P_s[::2][:len(st)], list(sig_s)[::2][:len(st)]
    a_s = ate(P_s, st.gt_poses)
    CMP.save_study(CMPD, "staer_slam", df_s); show(df_s, ["method", "ate_m", "rpe_m", "loops", "fallbacks", "ms_per_frame"])
    RESULTS["staer_slam"] = dict(config=best, ate_m=a_s["rmse"], drift_pct=100 * a_s["rmse"] / path_length(st.gt_poses),
                                 sweep=df_sweep.to_dict("records"), depth_pose_offset_cm=100 * st.depth_pose_offset_m)
    STUDIES.append(study("SLAM configuration sweep on the Staer warehouse", "first 6 s of the walkthrough; lower ATE is better",
                         df_sweep[["method", "ate_m", "rpe_m", "fallbacks", "ms_per_frame"]], None, highlight=best))
    save_results()

### B2 · Detections (computed once, reused by every study)

In [ ]:
if RUN_STAER:
    det_s = make_detector(STAER_CLASSES)
    frames_s = list(range(0, len(st), 3))
    torch.cuda.reset_peak_memory_stats()
    cache_s, det_ms_s = CMP.cache_detections(st, det_s, frames_s)
    CMP.save_cache(f"{OUT}/staer_detections.npz", cache_s)
    gpu_gb = torch.cuda.max_memory_allocated() / 2**30
    seg_kw = dict(min_score=SCORE_THRESHOLD, floor_classes=FLOOR_CLASSES)
    vis = visible_gt_assets(st, frames_s, max_depth=8.0)
    print(f"{sum(len(v) for v in cache_s.values())} detections in {len(frames_s)} keyframes · {det_ms_s:.0f} ms/keyframe · "
          f"GPU peak {gpu_gb:.1f} GB · detectable GT assets: {len(vis)}",
          pd.Series([st.instance_class[i] for i in vis]).value_counts().to_dict())

### B3 · Semantic fusion vs conventional methods (+ GASF ablations)
Same detections for every method. Baselines get the same ≥ 2-observation filter as GASF.
*Identity consistency* = share of ground-truth-labelled observations that end up in the single primary object of their asset.

In [ ]:
if RUN_STAER:
    pose_sets = {"ground-truth poses": (st.gt_poses, None),
                 "SLAM poses": ([a_s["T_align"] @ T for T in P_s], sig_s)}
    df_f, _ = CMP.fusion_study(st, cache_s, frames_s, STAER_CLASSES, pose_sets, seg_kwargs=seg_kw, max_depth=8.0)
    CMP.save_study(CMPD, "staer_fusion", df_f)
    cols = ["poses", "method", "objects", "precision", "recall", "f1", "duplicate_rate", "identity_consistency", "label_acc", "miou"]
    show(df_f, cols)
    g = df_f[df_f.poses == "ground-truth poses"]
    CMP.plot_methods(g, f"{FIGD}/staer_methods.png", title="Staer warehouse: fusion vs conventional methods (GT poses)")
    CMP.plot_ablation(g, f"{FIGD}/staer_ablation.png", "f1", "Staer: F1 of every method and GASF ablation")
    display(Image(f"{FIGD}/staer_methods.png")); display(Image(f"{FIGD}/staer_ablation.png"))
    STUDIES.append(study("Semantic fusion vs conventional methods (Staer warehouse)",
        f"{len(vis)} detectable assets · same detections for all methods · rows for GT and SLAM poses", df_f[cols], f"{FIGD}/staer_methods.png"))
    STUDIES.append(study("GASF component ablation (Staer, GT poses)", "F1 when one GASF component is removed, next to the baselines",
                         None, f"{FIGD}/staer_ablation.png"))
    RESULTS["staer_fusion"] = df_f.to_dict("records"); save_results()

### B4 · Robustness to pose error (what SLAM drift does to the twin)

In [ ]:
if RUN_STAER:
    df_n = CMP.pose_noise_study(st, cache_s, frames_s, STAER_CLASSES, levels=(0.0, 0.05, 0.1, 0.2, 0.4), seeds=(0, 1),
                                seg_kwargs=seg_kw)
    CMP.save_study(CMPD, "staer_pose_noise", df_n)
    show(df_n, ["pose_error_m", "method", "f1", "identity_consistency", "duplicate_rate", "objects"])
    CMP.plot_lines(df_n, "pose_error_m", "f1", f"{FIGD}/staer_noise_f1.png", "F1 vs pose drift", "end-of-run pose drift (m)", "F1")
    CMP.plot_lines(df_n, "pose_error_m", "duplicate_rate", f"{FIGD}/staer_noise_dup.png", "Duplicates per asset vs pose drift",
                   "end-of-run pose drift (m)", "duplicates per asset")
    display(Image(f"{FIGD}/staer_noise_f1.png")); display(Image(f"{FIGD}/staer_noise_dup.png"))
    STUDIES.append(study("Robustness to pose drift: F1", "random-walk drift added to GT poses, mean of 2 seeds", None, f"{FIGD}/staer_noise_f1.png"))
    STUDIES.append(study("Robustness to pose drift: duplicates", "lower is better", df_n[["pose_error_m","method","f1","duplicate_rate"]],
                         f"{FIGD}/staer_noise_dup.png"))
    RESULTS["staer_pose_noise"] = df_n.to_dict("records"); save_results()

### B5 · Sensitivity to the detector confidence threshold

In [ ]:
if RUN_STAER:
    df_th = CMP.threshold_study(st, cache_s, frames_s, STAER_CLASSES, thresholds=(0.20, 0.25, 0.30, 0.35, 0.40, 0.45),
                                seg_kwargs=dict(floor_classes=FLOOR_CLASSES))
    CMP.save_study(CMPD, "staer_threshold", df_th)
    show(df_th, ["threshold", "method", "precision", "recall", "f1", "objects"])
    CMP.plot_lines(df_th, "threshold", "f1", f"{FIGD}/staer_threshold.png", "F1 vs detector threshold", "box score threshold", "F1")
    display(Image(f"{FIGD}/staer_threshold.png"))
    STUDIES.append(study("Sensitivity to the detector threshold", "GT poses", df_th[["threshold","method","precision","recall","f1"]],
                         f"{FIGD}/staer_threshold.png"))
    RESULTS["staer_threshold"] = df_th.to_dict("records"); save_results()

### B6 · Warehouse digital twin + dashboard

In [ ]:
if RUN_STAER:
    out_s = run_mapping(st, st.gt_poses, None, None, STAER_CLASSES, gasf_stride=3, tsdf_voxel=0.05, tsdf_depth=10.0,
                        max_depth=8.0, det_cache=cache_s, seg_kwargs=seg_kw, map_sigma_max=0.15, verbose=False)
    twin_s = DigitalTwin("Staer scene03 twin"); store = TwinStore(TWIN_STORE)
    twin_s.update(out_s["gasf"].finalize(), STAER_CLASSES, f"{STAER_SCENE}-{STAER_WALK}", occ_new=out_s["occ"])
    store.push(twin_s)
    pc = out_s["tsdf"].point_cloud()
    perf = dict(ate_cm=100 * a_s["rmse"],
                stages_ms={"Detector + SAM 2.1 (T4, per keyframe)": det_ms_s, "SLAM tracking (CPU, per frame)": float(df_s.ms_per_frame.iloc[0]),
                           "3D lifting (per keyframe)": out_s["timings_ms"]["lift"], "GASF fusion (per keyframe)": out_s["timings_ms"]["fuse"],
                           "Map fusion (per frame)": out_s["timings_ms"]["tsdf"], "Occupancy (per keyframe)": out_s["timings_ms"]["occ"]},
                details={"GPU": RESULTS["env"]["gpu"], "Peak GPU memory": f"{gpu_gb:.1f} GB", "Peak RAM": f"{out_s['peak_ram_gb']:.1f} GB",
                         "Frames / keyframes": f"{len(st)} / {len(frames_s)}", "SLAM config": best,
                         "cam0 ↔ depth0 offset": f"{100*st.depth_pose_offset_m:.1f} cm", "Map points": f"{len(pc.points):,}"})
    build_dashboard(f"{OUT}/GeoSemTwinX_Dashboard.html", twin_s, np.asarray(pc.points), np.asarray(pc.colors),
                    traj_est=[T[:3, 3] for T in pose_sets["SLAM poses"][0]], traj_gt=[T[:3, 3] for T in st.gt_poses],
                    studies=STUDIES, perf=perf, title="GeoSemTwin-X Digital Twin", subtitle=f"Staer warehouse {STAER_SCENE} / walkthrough {STAER_WALK}")
    RESULTS["staer_twin"] = dict(assets=twin_s.counts(), n_assets=len(twin_s.present()), map_points=len(pc.points)); save_results()
    display(HTML(open(f"{OUT}/GeoSemTwinX_Dashboard.html").read()))

## Part C · Synthetic warehouse benchmark (optional, CPU, exact ground truth)
Includes change detection between two mapping sessions (2 moved, 2 removed, 2 added assets) and next-best-view exploration.

In [ ]:
from geosemtwinx import bench
if RUN_SYNTHETIC:
    bench.set_out(f"{OUT}/synthetic")
    bench.exp_slam(); bench.exp_semantic(seeds=(0,)); bench.exp_change(seeds=(0,)); bench.exp_nbv(seeds=(0,), budget=60.0)

## 4 · Download everything

In [ ]:
save_results()
!cd /content && zip -qr GeoSemTwinX_results.zip results
from google.colab import files
files.download("/content/GeoSemTwinX_results.zip")